# Time series explorer

Load transformed data and plot the time series for a selected **frequency band**.  
Uses **Dash** for the frequency dropdown and chart.  
Data is shown at hourly resolution (all hours for each day).

In [148]:
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

# Resolve project root (notebook may run from time_series/ or repo root)
_root = Path.cwd().resolve()
if _root.name == "time_series":
    _root = _root.parent
DATA_PATH = _root / "data" / "transformed" / "transformed_data.parquet"
if not DATA_PATH.exists():
    DATA_PATH = DATA_PATH.with_suffix(".csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Transform data not found at {DATA_PATH}. Run task transform first.")

df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
df = df.sort_values(["date", "hour"]).reset_index(drop=True)

# Frequency bands = all columns except date, hour
freq_bands = sorted([c for c in df.columns if c not in ("date", "hour")])
print(f"Loaded {len(df)} rows. Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Frequency bands: {len(freq_bands)}")

Loaded 9192 rows. Date range: 2025-01-01 to 2026-01-18
Frequency bands: 70


In [166]:
# Visualize: Dash app with frequency dropdown + plot (raw data)
import dash
from dash import dcc, html, Input, Output

app = dash.Dash(__name__, title="Time series explorer")
app.layout = html.Div([
    html.Div([
        html.Label("Frequency band:"),
        dcc.Dropdown(
            id="freq-band",
            options=[{"label": f, "value": f} for f in freq_bands],
            value=freq_bands[0],
            clearable=False,
            style={"width": "300px"},
        ),
    ], style={"margin": "10px"}),
    dcc.Graph(id="timeseries-plot", style={"height": "450px"}),
])

@app.callback(
    Output("timeseries-plot", "figure"),
    Input("freq-band", "value"),
)
def update_plot(freq_band):
    if freq_band not in df.columns:
        return go.Figure()
    sub = df[["date", "hour", freq_band]].dropna(subset=[freq_band]).copy()
    sub["datetime"] = pd.to_datetime(sub["date"]) + pd.to_timedelta(sub["hour"], unit="h")
    sub = sub.sort_values("datetime")
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=sub["datetime"],
            y=sub[freq_band],
            mode="lines+markers",
            name=freq_band,
            line=dict(width=1.5),
        )
    )
    fig.update_layout(
        title=f"Time series: {freq_band}",
        xaxis_title="Date & time",
        yaxis_title="Value",
        xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)),
        yaxis=dict(autorange=True, rangemode="normal"),
        margin=dict(b=80, t=60),
    )
    return fig

# Run: inline in notebook (dropdown + chart together). Use jupyter_mode="tab" to open in browser.
app.run(port=8052, jupyter_mode="inline", jupyter_height=500)

## Decomposition: trend, seasonality, and residual

The raw series can be split into:
- **Trend** — slow-moving level (centered moving average over the season length).
- **Seasonality** — repeating pattern: **per day** (24h) or **per week** (168h); same shape each day or each week.
- **Residual** — remainder (raw − trend − seasonal); short-term, irregular variation.

Additive model: **raw = trend + seasonal + residual**. Use the dropdowns to pick a **frequency band** and **seasonality period** (Daily 24h or Weekly 168h) and inspect the four panels below.

In [159]:
# Decomposition: trend, seasonality (daily or weekly), residual (additive). Period = 24h or 168h.
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose

# Ensure datetime column for indexing
if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")

def build_decomposition_figure(freq_band, period=24):
    """Decompose one band into trend, seasonal (per day or per week), and residual. period=24 or 168."""
    period = int(period)
    sub = df[["datetime", freq_band]].dropna(subset=[freq_band]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < 2 * period:  # need at least 2 full cycles
        return go.Figure()
    series = sub.set_index("datetime")[freq_band].asfreq("h")  # ensure hourly
    series = series.ffill().bfill()  # fill any gaps so decompose doesn't complain
    decomp = seasonal_decompose(series, model="additive", period=period, extrapolate_trend=min(period, 24))
    raw = decomp.observed
    trend = decomp.trend
    seasonal = decomp.seasonal
    residual = decomp.resid
    t = raw.index
    season_label = "Seasonality (per day)" if period == 24 else "Seasonality (per week)"
    fig = make_subplots(
        rows=4, cols=1,
        subplot_titles=("Raw", "Trend", season_label, "Residual"),
        shared_xaxes=True,
        vertical_spacing=0.06,
        row_heights=[0.3, 0.25, 0.25, 0.2],
    )
    fig.add_trace(go.Scatter(x=t, y=raw.values, mode="lines", name="Raw", line=dict(width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=trend.values, mode="lines", name="Trend", line=dict(width=1.5, color="#e74c3c")), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=seasonal.values, mode="lines", name="Seasonal", line=dict(width=1.2, color="#27ae60")), row=3, col=1)
    fig.add_trace(go.Scatter(x=t, y=residual.values, mode="lines", name="Noise", line=dict(width=1, color="#7f8c8d")), row=4, col=1)
    fig.update_layout(
        title=f"Decomposition: {freq_band} — {season_label}",
        height=700,
        margin=dict(b=60, t=60),
        showlegend=False,
    )
    fig.update_yaxes(title_text="Value", row=1, col=1)
    fig.update_yaxes(title_text="Trend", row=2, col=1)
    fig.update_yaxes(title_text="Seasonal", row=3, col=1)
    fig.update_yaxes(title_text="Residual", row=4, col=1)
    fig.update_xaxes(rangeslider=dict(visible=False))
    return fig

# Dash app: frequency dropdown → 4-panel decomposition
import dash
from dash import dcc, html, Input, Output

app_decomp = dash.Dash(__name__, title="Decomposition: trend, seasonality, residual")
app_decomp.layout = html.Div([
    html.Div([
        html.Label("Frequency band:"),
        dcc.Dropdown(
            id="decomp-freq",
            options=[{"label": f, "value": f} for f in freq_bands],
            value=freq_bands[0],
            clearable=False,
            style={"width": "300px"},
        ),
        html.Label("Seasonality period:", style={"marginLeft": "20px"}),
        dcc.Dropdown(
            id="decomp-period",
            options=[{"label": "Daily (24h)", "value": 24}, {"label": "Weekly (168h)", "value": 168}],
            value=24,
            clearable=False,
            style={"width": "180px", "marginLeft": "8px"},
        ),
    ], style={"margin": "10px", "display": "flex", "alignItems": "center", "flexWrap": "wrap", "gap": "8px"}),
    dcc.Graph(id="decomp-plot", style={"height": "720px"}),
])

@app_decomp.callback(
    Output("decomp-plot", "figure"),
    Input("decomp-freq", "value"),
    Input("decomp-period", "value"),
)
def update_decomp(freq_band, period):
    if not freq_band or freq_band not in df.columns:
        return go.Figure()
    return build_decomposition_figure(freq_band, period=period or 24)

app_decomp.run(port=8054, jupyter_mode="inline", jupyter_height=780)

In [160]:
# Naive forecasting (after looking at raw data above)
# One continuous time series per frequency band: hour-to-hour (all hours in chronological order).
print("Naive forecasting: next value = last observed.")
print("Formula: ŷ(t) = y(t-1) — for each time step (hour), prediction = value at previous hour.\n")

import numpy as np
import json

# One full time series per frequency band: (date + hour) sorted by datetime.
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
TEST_STEPS = 24 * 365  # last year of hours (use full year of data)
rows = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 1:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values
    for i in range(-TEST_STEPS, 0):
        rows.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": vals[i - 1]})

pred_df = pd.DataFrame(rows)
pred_df["datetime"] = pd.to_datetime(pred_df["datetime"])
print("Prediction values (sample — first 10 and last 5):")
print(pred_df.head(10).to_string(index=False))
print("...")
print(pred_df.tail(5).to_string(index=False))

# MAE and RMSE
y_true = pred_df["actual"].values
y_pred = pred_df["predicted"].values
mae_naive = float(np.mean(np.abs(y_true - y_pred)))
rmse_naive = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# MASE: scaling = naive MAE on the *evaluation set* (so MASE_naive = 1 by definition)
_mase_diffs = pred_df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_naive = float(mae_naive / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (naive):  {mae_naive:.4f}")
print(f"RMSE (naive): {rmse_naive:.4f}")
print(f"MASE (naive): {mase_naive:.4f}" if not np.isnan(mase_naive) else "MASE (naive): —")

# Store metrics
metrics_naive = {"mae": mae_naive, "rmse": rmse_naive, "mase": mase_naive, "n_predictions": len(pred_df), "test_steps_hours": TEST_STEPS}
out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_path = out_dir / "naive_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_naive, f, indent=2)
print(f"\nStored: {metrics_path}")
pred_df.to_parquet(out_dir / "naive_predictions.parquet", index=False)
print(f"Stored: {out_dir / 'naive_predictions.parquet'}")

# Plot: actual vs predicted over time + MAE/RMSE bar chart
naive_freqs = sorted(pred_df["frequency_band"].unique())
fig_metrics = go.Figure()
fig_metrics.add_trace(go.Bar(
    x=["MAE", "RMSE", "MASE"],
    y=[mae_naive, rmse_naive, mase_naive if not np.isnan(mase_naive) else 0],
    text=[f"{mae_naive:.4f}", f"{rmse_naive:.4f}", f"{mase_naive:.4f}" if not np.isnan(mase_naive) else "—"],
    textposition="outside",
    marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"],
))
fig_metrics.update_layout(title="Naive forecast: MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_naive = dash.Dash(__name__, title="Naive: actual vs predicted")
app_naive.layout = html.Div([
    html.Div([
        html.Label("Frequency band:"),
        dcc.Dropdown(id="naive-freq", options=[{"label": f, "value": f} for f in naive_freqs], value=naive_freqs[0], clearable=False, style={"width": "300px"}),
    ], style={"margin": "10px"}),
    dcc.Graph(id="naive-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}),
    dcc.Graph(id="metrics-bar", figure=fig_metrics),
])

@app_naive.callback(
    Output("naive-plot", "figure"),
    Input("naive-freq", "value"),
)
def update_naive_plot(freq):
    sub = pred_df[pred_df["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0:
        return go.Figure()
    x = sub["datetime"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=x, y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(
        title=f"Naive forecast: actual vs predicted — {freq} (hour-to-hour)",
        xaxis_title="Date & time",
        yaxis_title="Value",
        xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)),
        yaxis=dict(autorange=True, rangemode="normal"),
        margin=dict(b=80, t=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
    )
    return fig

app_naive.run(port=8053, jupyter_mode="inline", jupyter_height=900)

Naive forecasting: next value = last observed.
Formula: ŷ(t) = y(t-1) — for each time step (hour), prediction = value at previous hour.

Prediction values (sample — first 10 and last 5):
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  30.810811
2025-01-19 01:00:00    3.100-3.105 12.972973  15.202703
2025-01-19 02:00:00    3.100-3.105 10.337838  12.972973
2025-01-19 03:00:00    3.100-3.105  9.932432  10.337838
2025-01-19 04:00:00    3.100-3.105  8.716216   9.932432
2025-01-19 05:00:00    3.100-3.105 13.581081   8.716216
2025-01-19 06:00:00    3.100-3.105 16.013514  13.581081
2025-01-19 07:00:00    3.100-3.105 27.770270  16.013514
2025-01-19 08:00:00    3.100-3.105 28.581081  27.770270
2025-01-19 09:00:00    3.100-3.105 42.972973  28.581081
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  78.738739
2026-01-18 20:00:00    3.445-3.450 57.117117  78.738739
2026-01-18 21:00:00    3.

In [161]:
# Moving average forecasting (same continuous hour-to-hour series per frequency band)
print("Moving average: next value = mean of last W hours.")
print("Formula: ŷ(t) = mean(y(t-W), ..., y(t-1)).\n")

# Reuse datetime if present, else build
if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
TEST_STEPS_MA = 24 * 365  # last year of hours (use full year of data)
WINDOW = 2  # last 2 hours (tuned: closer to naive, less over-smoothing across daily cycle)
rows_ma = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS_MA + WINDOW:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values
    for i in range(-TEST_STEPS_MA, 0):
        pred = float(np.nanmean(vals[i - WINDOW : i]))
        rows_ma.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_ma = pd.DataFrame(rows_ma)
pred_df_ma["datetime"] = pd.to_datetime(pred_df_ma["datetime"])
print(f"Window = {WINDOW} hours. Prediction values (sample — first 10 and last 5):")
print(pred_df_ma.head(10).to_string(index=False))
print("...")
print(pred_df_ma.tail(5).to_string(index=False))

# MAE and RMSE
y_true_ma = pred_df_ma["actual"].values
y_pred_ma = pred_df_ma["predicted"].values
mae_ma = float(np.mean(np.abs(y_true_ma - y_pred_ma)))
rmse_ma = float(np.sqrt(np.mean((y_true_ma - y_pred_ma) ** 2)))

# MASE: scaling = naive MAE on this evaluation set (same as naive cell so comparable)
_mase_diffs = pred_df_ma.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_ma = float(mae_ma / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (moving avg):  {mae_ma:.4f}")
print(f"RMSE (moving avg): {rmse_ma:.4f}")
print(f"MASE (moving avg): {mase_ma:.4f}" if not np.isnan(mase_ma) else "MASE (moving avg): —")

# Store metrics
out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_ma = {"mae": mae_ma, "rmse": rmse_ma, "mase": mase_ma, "n_predictions": len(pred_df_ma), "test_steps_hours": TEST_STEPS_MA, "window_hours": WINDOW}
with open(out_dir / "moving_avg_metrics.json", "w") as f:
    json.dump(metrics_ma, f, indent=2)
pred_df_ma.to_parquet(out_dir / "moving_avg_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'moving_avg_metrics.json'}, {out_dir / 'moving_avg_predictions.parquet'}")

# Plot: actual vs predicted + MAE/RMSE bar chart
ma_freqs = sorted(pred_df_ma["frequency_band"].unique())
fig_metrics_ma = go.Figure()
fig_metrics_ma.add_trace(go.Bar(
    x=["MAE", "RMSE", "MASE"],
    y=[mae_ma, rmse_ma, mase_ma if not np.isnan(mase_ma) else 0],
    text=[f"{mae_ma:.4f}", f"{rmse_ma:.4f}", f"{mase_ma:.4f}" if not np.isnan(mase_ma) else "—"],
    textposition="outside",
    marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"],
))
fig_metrics_ma.update_layout(title="Moving average forecast: MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_ma = dash.Dash(__name__, title="Moving average: actual vs predicted")
app_ma.layout = html.Div([
    html.Div([
        html.Label("Frequency band:"),
        dcc.Dropdown(id="ma-freq", options=[{"label": f, "value": f} for f in ma_freqs], value=ma_freqs[0], clearable=False, style={"width": "300px"}),
    ], style={"margin": "10px"}),
    dcc.Graph(id="ma-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}),
    dcc.Graph(id="ma-metrics-bar", figure=fig_metrics_ma),
])

@app_ma.callback(
    Output("ma-plot", "figure"),
    Input("ma-freq", "value"),
)
def update_ma_plot(freq):
    sub = pred_df_ma[pred_df_ma["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0:
        return go.Figure()
    x = sub["datetime"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=x, y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(
        title=f"Moving average (W={WINDOW}h): actual vs predicted — {freq} (hour-to-hour)",
        xaxis_title="Date & time",
        yaxis_title="Value",
        xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)),
        yaxis=dict(autorange=True, rangemode="normal"),
        margin=dict(b=80, t=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
    )
    return fig

app_ma.run(port=8063, jupyter_mode="inline", jupyter_height=900)

Moving average: next value = mean of last W hours.
Formula: ŷ(t) = mean(y(t-W), ..., y(t-1)).

Window = 2 hours. Prediction values (sample — first 10 and last 5):
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  34.358108
2025-01-19 01:00:00    3.100-3.105 12.972973  23.006757
2025-01-19 02:00:00    3.100-3.105 10.337838  14.087838
2025-01-19 03:00:00    3.100-3.105  9.932432  11.655405
2025-01-19 04:00:00    3.100-3.105  8.716216  10.135135
2025-01-19 05:00:00    3.100-3.105 13.581081   9.324324
2025-01-19 06:00:00    3.100-3.105 16.013514  11.148649
2025-01-19 07:00:00    3.100-3.105 27.770270  14.797297
2025-01-19 08:00:00    3.100-3.105 28.581081  21.891892
2025-01-19 09:00:00    3.100-3.105 42.972973  28.175676
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  77.387387
2026-01-18 20:00:00    3.445-3.450 57.117117  78.738739
2026-01-18 21:00:00    3.445-3.450 79.099099  67.

In [162]:
# Exponential smoothing (same continuous hour-to-hour series per frequency band)
print("Exponential smoothing: next value = weighted average of past values (recent weighted more).")
print("Formula: level(t) = α·y(t) + (1-α)·level(t-1); ŷ(t+1) = level(t).\n")

if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
TEST_STEPS_ES = 24 * 365  # last year of hours
ALPHA = 0.98  # smoothing parameter (tuned: near-naive, slight smoothing of last observation)
rows_es = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS_ES + 1:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    # level[0] = vals[0]; level[j] = alpha*vals[j] + (1-alpha)*level[j-1]
    level = np.full(len(vals), np.nan)
    level[0] = vals[0]
    for j in range(1, len(vals)):
        level[j] = ALPHA * vals[j] + (1 - ALPHA) * level[j - 1]
    for i in range(-TEST_STEPS_ES, 0):
        pred = level[i - 1]
        rows_es.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_es = pd.DataFrame(rows_es)
pred_df_es["datetime"] = pd.to_datetime(pred_df_es["datetime"])
print(f"α = {ALPHA}. Prediction values (sample — first 10 and last 5):")
print(pred_df_es.head(10).to_string(index=False))
print("...")
print(pred_df_es.tail(5).to_string(index=False))

# MAE and RMSE
y_true_es = pred_df_es["actual"].values
y_pred_es = pred_df_es["predicted"].values
mae_es = float(np.mean(np.abs(y_true_es - y_pred_es)))
rmse_es = float(np.sqrt(np.mean((y_true_es - y_pred_es) ** 2)))

# MASE: scaling = naive MAE on this evaluation set (comparable across models)
_mase_diffs = pred_df_es.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_es = float(mae_es / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (exp smooth):  {mae_es:.4f}")
print(f"RMSE (exp smooth): {rmse_es:.4f}")
print(f"MASE (exp smooth): {mase_es:.4f}" if not np.isnan(mase_es) else "MASE (exp smooth): —")

# Store metrics
out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_es = {"mae": mae_es, "rmse": rmse_es, "mase": mase_es, "n_predictions": len(pred_df_es), "test_steps_hours": TEST_STEPS_ES, "alpha": ALPHA}
with open(out_dir / "exp_smooth_metrics.json", "w") as f:
    json.dump(metrics_es, f, indent=2)
pred_df_es.to_parquet(out_dir / "exp_smooth_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'exp_smooth_metrics.json'}, {out_dir / 'exp_smooth_predictions.parquet'}")

# Plot: actual vs predicted + MAE/RMSE bar chart
es_freqs = sorted(pred_df_es["frequency_band"].unique())
fig_metrics_es = go.Figure()
fig_metrics_es.add_trace(go.Bar(
    x=["MAE", "RMSE", "MASE"],
    y=[mae_es, rmse_es, mase_es if not np.isnan(mase_es) else 0],
    text=[f"{mae_es:.4f}", f"{rmse_es:.4f}", f"{mase_es:.4f}" if not np.isnan(mase_es) else "—"],
    textposition="outside",
    marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"],
))
fig_metrics_es.update_layout(title="Exponential smoothing forecast: MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_es = dash.Dash(__name__, title="Exponential smoothing: actual vs predicted")
app_es.layout = html.Div([
    html.Div([
        html.Label("Frequency band:"),
        dcc.Dropdown(id="es-freq", options=[{"label": f, "value": f} for f in es_freqs], value=es_freqs[0], clearable=False, style={"width": "300px"}),
    ], style={"margin": "10px"}),
    dcc.Graph(id="es-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}),
    dcc.Graph(id="es-metrics-bar", figure=fig_metrics_es),
])

@app_es.callback(
    Output("es-plot", "figure"),
    Input("es-freq", "value"),
)
def update_es_plot(freq):
    sub = pred_df_es[pred_df_es["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0:
        return go.Figure()
    x = sub["datetime"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=x, y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(
        title=f"Exponential smoothing (α={ALPHA}): actual vs predicted — {freq} (hour-to-hour)",
        xaxis_title="Date & time",
        yaxis_title="Value",
        xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)),
        yaxis=dict(autorange=True, rangemode="normal"),
        margin=dict(b=80, t=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
    )
    return fig

app_es.run(port=8055, jupyter_mode="inline", jupyter_height=900)

Exponential smoothing: next value = weighted average of past values (recent weighted more).
Formula: level(t) = α·y(t) + (1-α)·level(t-1); ŷ(t+1) = level(t).

α = 0.98. Prediction values (sample — first 10 and last 5):
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  30.954570
2025-01-19 01:00:00    3.100-3.105 12.972973  15.517740
2025-01-19 02:00:00    3.100-3.105 10.337838  13.023868
2025-01-19 03:00:00    3.100-3.105  9.932432  10.391558
2025-01-19 04:00:00    3.100-3.105  8.716216   9.941615
2025-01-19 05:00:00    3.100-3.105 13.581081   8.740724
2025-01-19 06:00:00    3.100-3.105 16.013514  13.484274
2025-01-19 07:00:00    3.100-3.105 27.770270  15.962929
2025-01-19 08:00:00    3.100-3.105 28.581081  27.534123
2025-01-19 09:00:00    3.100-3.105 42.972973  28.560142
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  78.683562
2026-01-18 20:00:00    3.445-3.450 57.117117  78.

In [163]:
# ARIMA forecasting: rolling 1-step-ahead (refit each hour, predict next only — same idea as naive/MA)
# So each prediction uses data up to the previous hour; fairer comparison and better accuracy. Slower.
print("ARIMA: autoregressive integrated moving average.")
print("Rolling 1-step-ahead: refit each hour, forecast next hour only. Order (p,d,q) = (1,0,1).\n")

from statsmodels.tsa.arima.model import ARIMA as _ARIMA

if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
# Rolling 1-step is slow (one fit per test hour per band). Use 3 days by default; 24*7 or 24*30 for longer runs.
TEST_STEPS_ARIMA = 24 * 3   # last 3 days of hours (increase to 24*7 for 7 days — much slower)
ARIMA_ORDER = (1, 0, 1)  # (p, d, q). Try e.g. (2,0,2), (1,1,1) for different fits
ARIMA_MAX_BANDS = 15  # limit bands for speed (None = all bands; 15 ≈ 1–2 min)
bands_to_run = (freq_bands[:ARIMA_MAX_BANDS] if ARIMA_MAX_BANDS else freq_bands)
rows_arima = []
_first_error = None
for freq in bands_to_run:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS_ARIMA + 50:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    n = len(vals)
    for i in range(n - TEST_STEPS_ARIMA, n - 1):
        train_series = pd.Series(vals[: i + 1]).dropna()
        if len(train_series) < 50:
            continue
        try:
            model = _ARIMA(train_series, order=ARIMA_ORDER)
            fitted = model.fit()
            f = fitted.get_forecast(steps=1)
            pred = float(f.predicted_mean.iloc[0])
        except Exception as e:
            if _first_error is None:
                _first_error = e
            pred = np.nan
        rows_arima.append({
            "datetime": dts[i + 1],
            "frequency_band": freq,
            "actual": vals[i + 1],
            "predicted": pred,
        })

if _first_error is not None and len(rows_arima) == 0:
    print(f"ARIMA fit/forecast failed for all bands. First error: {_first_error}")
pred_df_arima = pd.DataFrame(rows_arima)
if len(pred_df_arima) > 0:
    pred_df_arima["datetime"] = pd.to_datetime(pred_df_arima["datetime"])
else:
    pred_df_arima = pd.DataFrame(columns=["datetime", "frequency_band", "actual", "predicted"])
print(f"Order {ARIMA_ORDER}. Prediction values (sample — first 10 and last 5):")
print(pred_df_arima.head(10).to_string(index=False))
print("...")
print(pred_df_arima.tail(5).to_string(index=False))

# MAE and RMSE (ignore NaN predictions)
valid = pred_df_arima.dropna(subset=["predicted"])
y_true_arima = valid["actual"].values
y_pred_arima = valid["predicted"].values
mae_arima = float(np.mean(np.abs(y_true_arima - y_pred_arima))) if len(valid) > 0 else np.nan
rmse_arima = float(np.sqrt(np.mean((y_true_arima - y_pred_arima) ** 2))) if len(valid) > 0 else np.nan

# MASE: scaling = naive MAE on this evaluation set (comparable across models)
_mase_diffs = valid.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_arima = float(mae_arima / mase_denom) if (mase_denom and mase_denom > 0 and not np.isnan(mae_arima)) else np.nan

print(f"\nMAE (ARIMA):  {mae_arima:.4f}" if not np.isnan(mae_arima) else "\nMAE (ARIMA):  (no valid predictions)")
print(f"RMSE (ARIMA): {rmse_arima:.4f}" if not np.isnan(rmse_arima) else "RMSE (ARIMA): (no valid predictions)")
print(f"MASE (ARIMA): {mase_arima:.4f}" if not np.isnan(mase_arima) else "MASE (ARIMA): —")

# Store metrics
out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_arima = {"mae": mae_arima, "rmse": rmse_arima, "mase": mase_arima, "n_predictions": len(valid), "test_steps_hours": TEST_STEPS_ARIMA, "order": list(ARIMA_ORDER)}
with open(out_dir / "arima_metrics.json", "w") as f:
    json.dump(metrics_arima, f, indent=2)
pred_df_arima.to_parquet(out_dir / "arima_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'arima_metrics.json'}, {out_dir / 'arima_predictions.parquet'}")

# Plot: actual vs predicted + MAE/RMSE bar chart
arima_freqs = sorted(pred_df_arima["frequency_band"].unique())
if not arima_freqs:
    arima_freqs = ["(no successful fits)"]
arima_options = [{"label": f, "value": f} for f in arima_freqs]
arima_default = arima_freqs[0]

fig_metrics_arima = go.Figure()
fig_metrics_arima.add_trace(go.Bar(
    x=["MAE", "RMSE", "MASE"],
    y=[mae_arima if not np.isnan(mae_arima) else 0, rmse_arima if not np.isnan(rmse_arima) else 0, mase_arima if not np.isnan(mase_arima) else 0],
    text=[f"{mae_arima:.4f}" if not np.isnan(mae_arima) else "—", f"{rmse_arima:.4f}" if not np.isnan(rmse_arima) else "—", f"{mase_arima:.4f}" if not np.isnan(mase_arima) else "—"],
    textposition="outside",
    marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"],
))
fig_metrics_arima.update_layout(title="ARIMA forecast: MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_arima = dash.Dash(__name__, title="ARIMA: actual vs predicted")
app_arima.layout = html.Div([
    html.Div([
        html.Label("Frequency band:"),
        dcc.Dropdown(id="arima-freq", options=arima_options, value=arima_default, clearable=False, style={"width": "300px"}),
    ], style={"margin": "10px"}),
    dcc.Graph(id="arima-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}),
    dcc.Graph(id="arima-metrics-bar", figure=fig_metrics_arima),
])

@app_arima.callback(
    Output("arima-plot", "figure"),
    Input("arima-freq", "value"),
)
def update_arima_plot(freq):
    if freq is None or freq == "(no successful fits)" or len(pred_df_arima) == 0:
        return go.Figure()
    sub = pred_df_arima[pred_df_arima["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0:
        return go.Figure()
    x = sub["datetime"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=x, y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(
        title=f"ARIMA {ARIMA_ORDER}: actual vs predicted — {freq} (hour-to-hour)",
        xaxis_title="Date & time",
        yaxis_title="Value",
        xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)),
        yaxis=dict(autorange=True, rangemode="normal"),
        margin=dict(b=80, t=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
    )
    return fig

app_arima.run(port=8056, jupyter_mode="inline", jupyter_height=900)

ARIMA: autoregressive integrated moving average.
Rolling 1-step-ahead: refit each hour, forecast next hour only. Order (p,d,q) = (1,0,1).

Order (1, 0, 1). Prediction values (sample — first 10 and last 5):
           datetime frequency_band    actual  predicted
2026-01-16 01:00:00    3.100-3.105 16.036036  18.214936
2026-01-16 02:00:00    3.100-3.105 16.936937  16.368175
2026-01-16 03:00:00    3.100-3.105 16.756757  17.129458
2026-01-16 04:00:00    3.100-3.105 16.936937  16.984521
2026-01-16 05:00:00    3.100-3.105 19.459459  17.140965
2026-01-16 06:00:00    3.100-3.105 31.171171  19.372466
2026-01-16 07:00:00    3.100-3.105 39.279279  29.762076
2026-01-16 08:00:00    3.100-3.105 49.009009  37.122825
2026-01-16 09:00:00    3.100-3.105 50.810811  45.862873
2026-01-16 10:00:00    3.100-3.105 56.216216  47.617560
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.170-3.175 11.171171  11.125457
2026-01-18 20:00:00    3.170-3.175 52.612613  11.907425
2026-0

In [164]:
# Amazon Chronos forecasting (pretrained time-series foundation model)
# https://huggingface.co/collections/amazon/chronos-models-and-datasets — install: pip install chronos-forecasting
# If you see pydevd/ipywidget errors: run this cell with "Run" (not "Debug") and ensure the kernel uses .venv.

import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

print("Chronos: Amazon's pretrained time-series foundation model (Chronos-2).")
print("Rolling forecasts: back-to-back 72h windows (512h context each). Full coverage, no gaps.\n")

CHRONOS_AVAILABLE = False
try:
    from chronos import BaseChronosPipeline
    CHRONOS_AVAILABLE = True
except ImportError as e:
    print(f"Chronos not installed: {e}")
    print("Install: pip install chronos-forecasting")
    print("Skipping Chronos cell.\n")

if CHRONOS_AVAILABLE:
    if "datetime" not in df.columns:
        df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
    TEST_STEPS_CHRONOS = 72   # forecast next 72 hours (3 days) per window
    MIN_CONTEXT = 512         # history length per window (Chronos-2 supports long context)
    STRIDE_HOURS = TEST_STEPS_CHRONOS  # back-to-back 72h windows so graph has no gaps
    context_dfs = []
    actuals_by_band = {}
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        n = len(sub)
        if n < TEST_STEPS_CHRONOS + MIN_CONTEXT:
            continue
        # Rolling windows: at each step t we use [t-MIN_CONTEXT:t] as context, predict [t:t+TEST_STEPS]
        for t in range(MIN_CONTEXT, n - TEST_STEPS_CHRONOS + 1, STRIDE_HOURS):
            context_part = sub.iloc[t - MIN_CONTEXT : t]
            window_id = f"{freq}_t{t}"
            context_dfs.append(pd.DataFrame({
                "item_id": window_id,
                "timestamp": context_part["datetime"].values,
                "target": context_part[freq].values.astype(np.float64),
            }))
            actuals_by_band[window_id] = sub.iloc[t : t + TEST_STEPS_CHRONOS][["datetime", freq]].rename(columns={freq: "actual"})

if CHRONOS_AVAILABLE and len(context_dfs) > 0:
    context_df = pd.concat(context_dfs, ignore_index=True)
    context_df["timestamp"] = pd.to_datetime(context_df["timestamp"])
    try:
        pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
        chronos_pred = pipeline.predict_df(
            context_df,
            prediction_length=TEST_STEPS_CHRONOS,
            quantile_levels=[0.5],
            id_column="item_id",
            timestamp_column="timestamp",
            target="target",
        )
    except Exception as e:
        print(f"Failed to load or run Chronos model: {e}")
        chronos_pred = None
else:
    chronos_pred = None

if chronos_pred is not None and len(chronos_pred) > 0:
    rows_ch = []
    for window_id in chronos_pred["item_id"].unique():
        if window_id not in actuals_by_band:
            continue
        # window_id is "{freq}_t{t}"; frequency_band is the part before "_t"
        frequency_band = window_id.rsplit("_t", 1)[0] if "_t" in window_id else window_id
        actuals = actuals_by_band[window_id].set_index("datetime")["actual"]
        sub = chronos_pred[chronos_pred["item_id"] == window_id]
        for _, row in sub.iterrows():
            ts = pd.Timestamp(row["timestamp"])
            pred_val = row["predictions"] if "predictions" in row else row.get("0.5", np.nan)
            actual_val = actuals.get(ts, np.nan) if ts in actuals.index else np.nan
            rows_ch.append({"datetime": ts, "frequency_band": frequency_band, "actual": actual_val, "predicted": float(pred_val) if pd.notna(pred_val) else np.nan})
    pred_df_tf = pd.DataFrame(rows_ch)
    pred_df_tf["datetime"] = pd.to_datetime(pred_df_tf["datetime"])
else:
    pred_df_tf = pd.DataFrame(columns=["datetime", "frequency_band", "actual", "predicted"])

if len(pred_df_tf) > 0:
    print("Prediction values (sample — first 10 and last 5):")
    print(pred_df_tf.head(10).to_string(index=False))
    print("...")
    print(pred_df_tf.tail(5).to_string(index=False))
    valid_tf = pred_df_tf.dropna(subset=["predicted"])
    y_true_tf = valid_tf["actual"].values
    y_pred_tf = valid_tf["predicted"].values
    mae_tf = float(np.mean(np.abs(y_true_tf - y_pred_tf)))
    rmse_tf = float(np.sqrt(np.mean((y_true_tf - y_pred_tf) ** 2)))

    # MASE: scaling = naive MAE on this evaluation set (comparable across models)
    _mase_diffs = pred_df_tf.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
    mase_tf = float(mae_tf / mase_denom) if mase_denom and mase_denom > 0 else np.nan

    print(f"\nMAE (Chronos):  {mae_tf:.4f}")
    print(f"RMSE (Chronos): {rmse_tf:.4f}")
    print(f"MASE (Chronos): {mase_tf:.4f}" if not np.isnan(mase_tf) else "MASE (Chronos): —")
    out_dir = _root / "data" / "time_series"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "chronos_metrics.json", "w") as f:
        json.dump({"mae": mae_tf, "rmse": rmse_tf, "mase": mase_tf, "n_predictions": len(valid_tf), "horizon": TEST_STEPS_CHRONOS}, f, indent=2)
    pred_df_tf.to_parquet(out_dir / "chronos_predictions.parquet", index=False)
    print(f"\nStored: {out_dir / 'chronos_metrics.json'}, {out_dir / 'chronos_predictions.parquet'}")

    tf_freqs = sorted(pred_df_tf["frequency_band"].unique())
    fig_metrics_tf = go.Figure()
    fig_metrics_tf.add_trace(go.Bar(x=["MAE", "RMSE", "MASE"], y=[mae_tf, rmse_tf, mase_tf if not np.isnan(mase_tf) else 0], text=[f"{mae_tf:.4f}", f"{rmse_tf:.4f}", f"{mase_tf:.4f}" if not np.isnan(mase_tf) else "—"], textposition="outside", marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"]))
    fig_metrics_tf.update_layout(title="Chronos forecast: MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)
    app_tf = dash.Dash(__name__, title="Chronos: actual vs predicted")
    app_tf.layout = html.Div([
        html.Div([html.Label("Frequency band:"), dcc.Dropdown(id="tf-freq", options=[{"label": f, "value": f} for f in tf_freqs], value=tf_freqs[0], clearable=False, style={"width": "300px"})], style={"margin": "10px"}),
        dcc.Graph(id="tf-plot", style={"height": "450px"}),
        html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}), dcc.Graph(id="tf-metrics-bar", figure=fig_metrics_tf),
    ])
    @app_tf.callback(Output("tf-plot", "figure"), Input("tf-freq", "value"))
    def update_tf_plot(freq):
        sub = pred_df_tf[pred_df_tf["frequency_band"] == freq].sort_values("datetime")
        if len(sub) == 0: return go.Figure()
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
        fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
        fig.update_layout(title=f"Chronos: actual vs predicted — {freq}", xaxis_title="Date & time", yaxis_title="Value", xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)), yaxis=dict(autorange=True, rangemode="normal"), margin=dict(b=80, t=60), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), hovermode="x unified")
        return fig
    app_tf.run(port=8057, jupyter_mode="inline", jupyter_height=900)
else:
    print("No Chronos predictions (install chronos-forecasting and re-run, or all bands failed).")

Chronos: Amazon's pretrained time-series foundation model (Chronos-2).
Rolling forecasts: back-to-back 72h windows (512h context each). Full coverage, no gaps.

Prediction values (sample — first 10 and last 5):
           datetime frequency_band    actual  predicted
2025-01-22 08:00:00    3.100-3.105 43.986486  34.729458
2025-01-22 09:00:00    3.100-3.105 35.067568  34.130676
2025-01-22 10:00:00    3.100-3.105 38.513514  38.617661
2025-01-22 11:00:00    3.100-3.105 36.283784  43.904869
2025-01-22 12:00:00    3.100-3.105 49.256757  51.996376
2025-01-22 13:00:00    3.100-3.105 36.891892  47.764740
2025-01-22 14:00:00    3.100-3.105 36.486486  47.340996
2025-01-22 15:00:00    3.100-3.105 42.972973  51.451797
2025-01-22 16:00:00    3.100-3.105 46.621622  61.259979
2025-01-22 17:00:00    3.100-3.105 47.432432  62.796730
...
           datetime frequency_band    actual  predicted
2026-01-17 03:00:00    3.445-3.450 49.009009  69.370636
2026-01-17 04:00:00    3.445-3.450 47.207207  68.951515
2

In [114]:
# Seasonal naive: yesterday at this time. ŷ(t) = y(t-24)
print("Seasonal naive: prediction = same hour yesterday.")
print("Formula: ŷ(t) = y(t-24).\n")

if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
TEST_STEPS_SN = 24 * 365
SEASON_HOURS = 24  # one day
rows_sn = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS_SN + SEASON_HOURS:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    for i in range(-TEST_STEPS_SN, 0):
        rows_sn.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": vals[i - SEASON_HOURS]})

pred_df_sn = pd.DataFrame(rows_sn)
pred_df_sn["datetime"] = pd.to_datetime(pred_df_sn["datetime"])
print("Prediction values (sample — first 10 and last 5):")
print(pred_df_sn.head(10).to_string(index=False))
print("...")
print(pred_df_sn.tail(5).to_string(index=False))

y_true_sn = pred_df_sn["actual"].values
y_pred_sn = pred_df_sn["predicted"].values
mae_sn = float(np.mean(np.abs(y_true_sn - y_pred_sn)))
rmse_sn = float(np.sqrt(np.mean((y_true_sn - y_pred_sn) ** 2)))
_mase_diffs = pred_df_sn.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_sn = float(mae_sn / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (seasonal naive):  {mae_sn:.4f}")
print(f"RMSE (seasonal naive): {rmse_sn:.4f}")
print(f"MASE (seasonal naive): {mase_sn:.4f}" if not np.isnan(mase_sn) else "MASE (seasonal naive): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_sn = {"mae": mae_sn, "rmse": rmse_sn, "mase": mase_sn, "n_predictions": len(pred_df_sn), "test_steps_hours": TEST_STEPS_SN, "season_hours": SEASON_HOURS}
with open(out_dir / "seasonal_naive_metrics.json", "w") as f:
    json.dump(metrics_sn, f, indent=2)
pred_df_sn.to_parquet(out_dir / "seasonal_naive_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'seasonal_naive_metrics.json'}, {out_dir / 'seasonal_naive_predictions.parquet'}")

sn_freqs = sorted(pred_df_sn["frequency_band"].unique())
fig_metrics_sn = go.Figure()
fig_metrics_sn.add_trace(go.Bar(x=["MAE", "RMSE", "MASE"], y=[mae_sn, rmse_sn, mase_sn if not np.isnan(mase_sn) else 0], text=[f"{mae_sn:.4f}", f"{rmse_sn:.4f}", f"{mase_sn:.4f}" if not np.isnan(mase_sn) else "—"], textposition="outside", marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"]))
fig_metrics_sn.update_layout(title="Seasonal naive (yesterday): MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_sn = dash.Dash(__name__, title="Seasonal naive: actual vs predicted")
app_sn.layout = html.Div([
    html.Div([html.Label("Frequency band:"), dcc.Dropdown(id="sn-freq", options=[{"label": f, "value": f} for f in sn_freqs], value=sn_freqs[0], clearable=False, style={"width": "300px"})], style={"margin": "10px"}),
    dcc.Graph(id="sn-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}), dcc.Graph(id="sn-metrics-bar", figure=fig_metrics_sn),
])
@app_sn.callback(Output("sn-plot", "figure"), Input("sn-freq", "value"))
def update_sn_plot(freq):
    sub = pred_df_sn[pred_df_sn["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0: return go.Figure()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(title=f"Seasonal naive (yesterday): actual vs predicted — {freq}", xaxis_title="Date & time", yaxis_title="Value", xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)), yaxis=dict(autorange=True, rangemode="normal"), margin=dict(b=80, t=60), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), hovermode="x unified")
    return fig
app_sn.run(port=8058, jupyter_mode="inline", jupyter_height=900)

Seasonal naive: prediction = same hour yesterday.
Formula: ŷ(t) = y(t-24).

Prediction values (sample — first 10 and last 5):
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  17.432432
2025-01-19 01:00:00    3.100-3.105 12.972973  10.337838
2025-01-19 02:00:00    3.100-3.105 10.337838  10.337838
2025-01-19 03:00:00    3.100-3.105  9.932432   8.716216
2025-01-19 04:00:00    3.100-3.105  8.716216   9.932432
2025-01-19 05:00:00    3.100-3.105 13.581081  11.756757
2025-01-19 06:00:00    3.100-3.105 16.013514  12.162162
2025-01-19 07:00:00    3.100-3.105 27.770270  13.581081
2025-01-19 08:00:00    3.100-3.105 28.581081  22.500000
2025-01-19 09:00:00    3.100-3.105 42.972973  33.243243
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  53.513514
2026-01-18 20:00:00    3.445-3.450 57.117117  57.837838
2026-01-18 21:00:00    3.445-3.450 79.099099  57.477477
2026-01-18 22:00:00    3.445-3

In [131]:
# Hybrid: blend of naive (last hour) + seasonal naive (same hour yesterday). Uses both persistence and daily seasonality.
# ŷ(t) = w * y(t-1) + (1-w) * y(t-24). Tuned w=0.7 to favor last hour (strong) while adding same-hour-yesterday.
print("Hybrid: 70% last hour + 30% same hour yesterday.")
print("Formula: ŷ(t) = 0.7 * y(t-1) + 0.3 * y(t-24).\n")

if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
TEST_STEPS_HY = 24 * 365
W_NAIVE = 0.7   # weight on y(t-1); (1 - W_NAIVE) on y(t-24)
rows_hy = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS_HY + 24:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    for i in range(-TEST_STEPS_HY, 0):
        pred = W_NAIVE * vals[i - 1] + (1 - W_NAIVE) * vals[i - 24]
        rows_hy.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_hy = pd.DataFrame(rows_hy)
pred_df_hy["datetime"] = pd.to_datetime(pred_df_hy["datetime"])
print("Prediction values (sample — first 10 and last 5):")
print(pred_df_hy.head(10).to_string(index=False))
print("...")
print(pred_df_hy.tail(5).to_string(index=False))

y_true_hy = pred_df_hy["actual"].values
y_pred_hy = pred_df_hy["predicted"].values
mae_hy = float(np.mean(np.abs(y_true_hy - y_pred_hy)))
rmse_hy = float(np.sqrt(np.mean((y_true_hy - y_pred_hy) ** 2)))
_mase_diffs = pred_df_hy.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_hy = float(mae_hy / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (hybrid):  {mae_hy:.4f}")
print(f"RMSE (hybrid): {rmse_hy:.4f}")
print(f"MASE (hybrid): {mase_hy:.4f}" if not np.isnan(mase_hy) else "MASE (hybrid): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_hy = {"mae": mae_hy, "rmse": rmse_hy, "mase": mase_hy, "n_predictions": len(pred_df_hy), "test_steps_hours": TEST_STEPS_HY, "w_naive": W_NAIVE}
with open(out_dir / "hybrid_metrics.json", "w") as f:
    json.dump(metrics_hy, f, indent=2)
pred_df_hy.to_parquet(out_dir / "hybrid_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'hybrid_metrics.json'}, {out_dir / 'hybrid_predictions.parquet'}")

hy_freqs = sorted(pred_df_hy["frequency_band"].unique())
fig_metrics_hy = go.Figure()
fig_metrics_hy.add_trace(go.Bar(x=["MAE", "RMSE", "MASE"], y=[mae_hy, rmse_hy, mase_hy if not np.isnan(mase_hy) else 0], text=[f"{mae_hy:.4f}", f"{rmse_hy:.4f}", f"{mase_hy:.4f}" if not np.isnan(mase_hy) else "—"], textposition="outside", marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"]))
fig_metrics_hy.update_layout(title="Hybrid (0.7*naive+0.3*seasonal): MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_hy = dash.Dash(__name__, title="Hybrid: actual vs predicted")
app_hy.layout = html.Div([
    html.Div([html.Label("Frequency band:"), dcc.Dropdown(id="hy-freq", options=[{"label": f, "value": f} for f in hy_freqs], value=hy_freqs[0], clearable=False, style={"width": "300px"})], style={"margin": "10px"}),
    dcc.Graph(id="hy-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}), dcc.Graph(id="hy-metrics-bar", figure=fig_metrics_hy),
])
@app_hy.callback(Output("hy-plot", "figure"), Input("hy-freq", "value"))
def update_hy_plot(freq):
    sub = pred_df_hy[pred_df_hy["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0: return go.Figure()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(title=f"Hybrid (0.7*last_h + 0.3*same_h_yesterday): actual vs predicted — {freq}", xaxis_title="Date & time", yaxis_title="Value", xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)), yaxis=dict(autorange=True, rangemode="normal"), margin=dict(b=80, t=60), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), hovermode="x unified")
    return fig
app_hy.run(port=8062, jupyter_mode="inline", jupyter_height=900)

Hybrid: 70% last hour + 30% same hour yesterday.
Formula: ŷ(t) = 0.7 * y(t-1) + 0.3 * y(t-24).

Prediction values (sample — first 10 and last 5):
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  26.797297
2025-01-19 01:00:00    3.100-3.105 12.972973  13.743243
2025-01-19 02:00:00    3.100-3.105 10.337838  12.182432
2025-01-19 03:00:00    3.100-3.105  9.932432   9.851351
2025-01-19 04:00:00    3.100-3.105  8.716216   9.932432
2025-01-19 05:00:00    3.100-3.105 13.581081   9.628378
2025-01-19 06:00:00    3.100-3.105 16.013514  13.155405
2025-01-19 07:00:00    3.100-3.105 27.770270  15.283784
2025-01-19 08:00:00    3.100-3.105 28.581081  26.189189
2025-01-19 09:00:00    3.100-3.105 42.972973  29.979730
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  71.171171
2026-01-18 20:00:00    3.445-3.450 57.117117  72.468468
2026-01-18 21:00:00    3.445-3.450 79.099099  57.225225
2026-01-18

In [137]:
# OLS blend: data-driven weights for naive + seasonal naive. Fit ŷ = a*y(t-1) + b*y(t-24) + c on train, predict on test.
print("OLS blend: fit ŷ = a*y(t-1) + b*y(t-24) + c on training data (pre–test period), apply to test.\n")

if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
# Use 350 days test so with ~9192 rows we have enough history (need test + train_min + 24 <= n)
TEST_STEPS_OLS = 24 * 350
TRAIN_MIN = 24 * 30  # at least 30 days of train history per band
rows_ols = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    n = len(sub)
    if n < TEST_STEPS_OLS + TRAIN_MIN + 24:
        continue
    vals = sub[freq].values.astype(float)
    dts = sub["datetime"].values
    # Train: indices [24 : n - TEST_STEPS_OLS] so we have y(t-1), y(t-24), y(t)
    train_end = n - TEST_STEPS_OLS
    if train_end - 24 < TRAIN_MIN:
        continue
    y_train = vals[24:train_end]
    x1 = vals[23:train_end - 1]   # y(t-1)
    x24 = vals[0:train_end - 24]  # y(t-24)
    X_train = np.column_stack([x1, x24])
    # Fit: y = a*x1 + b*x24 (no intercept to keep scale; or with intercept)
    X_train = np.column_stack([np.ones_like(y_train), X_train])
    try:
        coeffs, _, _, _ = np.linalg.lstsq(X_train, y_train, rcond=None)
        c, a, b = coeffs[0], coeffs[1], coeffs[2]
    except Exception:
        a, b, c = 1.0, 0.0, 0.0  # fallback to naive
    for i in range(-TEST_STEPS_OLS, 0):
        pred = c + a * vals[i - 1] + b * vals[i - 24]
        rows_ols.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_ols = pd.DataFrame(rows_ols)
if len(pred_df_ols) == 0:
    pred_df_ols = pd.DataFrame(columns=["datetime", "frequency_band", "actual", "predicted"])
    print("No OLS predictions: no frequency band had enough history (need >= TEST_STEPS_OLS + TRAIN_MIN + 24 hours).")
else:
    pred_df_ols["datetime"] = pd.to_datetime(pred_df_ols["datetime"])
print("Sample:")
print(pred_df_ols.head(10).to_string(index=False))
print("...")
print(pred_df_ols.tail(5).to_string(index=False))

if len(pred_df_ols) > 0:
    y_true_ols = pred_df_ols["actual"].values
    y_pred_ols = pred_df_ols["predicted"].values
    mae_ols = float(np.mean(np.abs(y_true_ols - y_pred_ols)))
    rmse_ols = float(np.sqrt(np.mean((y_true_ols - y_pred_ols) ** 2)))
    _mase_diffs = pred_df_ols.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
    mase_ols = float(mae_ols / mase_denom) if mase_denom and mase_denom > 0 else np.nan
else:
    mae_ols = rmse_ols = mase_ols = None

print(f"\nMAE (OLS blend):  {mae_ols:.4f}" if mae_ols is not None else "\nMAE (OLS blend): —")
print(f"RMSE (OLS blend): {rmse_ols:.4f}" if rmse_ols is not None else "RMSE (OLS blend): —")
print(f"MASE (OLS blend): {mase_ols:.4f}" if mase_ols is not None else "MASE (OLS blend): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "ols_blend_metrics.json", "w") as f:
    json.dump({"mae": mae_ols, "rmse": rmse_ols, "mase": mase_ols, "n_predictions": len(pred_df_ols)}, f, indent=2)
pred_df_ols.to_parquet(out_dir / "ols_blend_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'ols_blend_metrics.json'}, {out_dir / 'ols_blend_predictions.parquet'}")

OLS blend: fit ŷ = a*y(t-1) + b*y(t-24) + c on training data (pre–test period), apply to test.

Sample:
           datetime frequency_band    actual  predicted
2025-02-03 00:00:00    3.100-3.105  8.828829   9.935228
2025-02-03 01:00:00    3.100-3.105  7.747748   9.409469
2025-02-03 02:00:00    3.100-3.105  7.747748   8.918371
2025-02-03 03:00:00    3.100-3.105  7.747748   8.646827
2025-02-03 04:00:00    3.100-3.105  7.747748   8.646827
2025-02-03 05:00:00    3.100-3.105  8.828829   8.646827
2025-02-03 06:00:00    3.100-3.105 10.090090   9.409469
2025-02-03 07:00:00    3.100-3.105 16.756757  10.932823
2025-02-03 08:00:00    3.100-3.105 22.162162  16.043103
2025-02-03 09:00:00    3.100-3.105 27.927928  20.806722
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  68.671543
2026-01-18 20:00:00    3.445-3.450 57.117117  69.747291
2026-01-18 21:00:00    3.445-3.450 79.099099  54.742999
2026-01-18 22:00:00    3.445-3.450 79.099099  67.799

In [134]:
# Median ensemble: pred = median(naive, seasonal_naive). Robust to outliers in either forecast.
print("Median ensemble: ŷ(t) = median( y(t-1), y(t-24) ).\n")

TEST_STEPS_MED = 24 * 365
rows_med = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS_MED + 24:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    for i in range(-TEST_STEPS_MED, 0):
        pred = float(np.median([vals[i - 1], vals[i - 24]]))
        rows_med.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_med = pd.DataFrame(rows_med)
pred_df_med["datetime"] = pd.to_datetime(pred_df_med["datetime"])
print("Sample:")
print(pred_df_med.head(10).to_string(index=False))
print("...")
print(pred_df_med.tail(5).to_string(index=False))

y_true_med = pred_df_med["actual"].values
y_pred_med = pred_df_med["predicted"].values
mae_med = float(np.mean(np.abs(y_true_med - y_pred_med)))
rmse_med = float(np.sqrt(np.mean((y_true_med - y_pred_med) ** 2)))
_mase_diffs = pred_df_med.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_med = float(mae_med / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (median ensemble):  {mae_med:.4f}")
print(f"RMSE (median ensemble): {rmse_med:.4f}")
print(f"MASE (median ensemble): {mase_med:.4f}" if not np.isnan(mase_med) else "MASE (median ensemble): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "median_ensemble_metrics.json", "w") as f:
    json.dump({"mae": mae_med, "rmse": rmse_med, "mase": mase_med, "n_predictions": len(pred_df_med)}, f, indent=2)
pred_df_med.to_parquet(out_dir / "median_ensemble_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'median_ensemble_metrics.json'}, {out_dir / 'median_ensemble_predictions.parquet'}")

Median ensemble: ŷ(t) = median( y(t-1), y(t-24) ).

Sample:
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  24.121622
2025-01-19 01:00:00    3.100-3.105 12.972973  12.770270
2025-01-19 02:00:00    3.100-3.105 10.337838  11.655405
2025-01-19 03:00:00    3.100-3.105  9.932432   9.527027
2025-01-19 04:00:00    3.100-3.105  8.716216   9.932432
2025-01-19 05:00:00    3.100-3.105 13.581081  10.236486
2025-01-19 06:00:00    3.100-3.105 16.013514  12.871622
2025-01-19 07:00:00    3.100-3.105 27.770270  14.797297
2025-01-19 08:00:00    3.100-3.105 28.581081  25.135135
2025-01-19 09:00:00    3.100-3.105 42.972973  30.912162
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  66.126126
2026-01-18 20:00:00    3.445-3.450 57.117117  68.288288
2026-01-18 21:00:00    3.445-3.450 79.099099  57.297297
2026-01-18 22:00:00    3.445-3.450 79.099099  64.054054
2026-01-18 23:00:00    3.445-3.450 78.19

**Tuned blend (validation-tuned α)**  
ŷ = α·y(t-1) + (1-α)·y(t-24). α is chosen per band by minimizing MAE on a short validation period (last 14 days before test). Grid: α in [0.97, 1.0] so the model stays naive-heavy; this can beat pure naive when a small seasonal component helps.

In [141]:
# Tuned blend: ŷ = α*y(t-1) + (1-α)*y(t-24). α tuned per band on validation (min MAE), grid [0.97, 1.0].
print("Tuned blend (validation-tuned α): ŷ = α·y(t-1) + (1-α)·y(t-24), α in [0.97, 1.0] per band.\n")

if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
TEST_STEPS_TB = 24 * 365
VAL_STEPS_TB = 24 * 14   # 14 days validation (fits in 9192 rows: 8760+336+24=9120)
TRAIN_MIN_TB = 24 * 14
rows_tb = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    n = len(sub)
    if n < TEST_STEPS_TB + TRAIN_MIN_TB + 24:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    val_start = n - TEST_STEPS_TB - VAL_STEPS_TB
    val_end = n - TEST_STEPS_TB
    if val_end - 24 <= val_start:
        continue
    best_mae = np.inf
    best_alpha = 1.0
    for alpha in np.linspace(0.97, 1.0, 31):
        preds = alpha * vals[val_start + 23 : val_end - 1] + (1 - alpha) * vals[val_start : val_end - 24]
        actuals = vals[val_start + 24 : val_end]
        mae = np.mean(np.abs(actuals - preds))
        if mae < best_mae:
            best_mae = mae
            best_alpha = alpha
    for i in range(-TEST_STEPS_TB, 0):
        pred = best_alpha * vals[i - 1] + (1 - best_alpha) * vals[i - 24]
        rows_tb.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_tb = pd.DataFrame(rows_tb)
if len(pred_df_tb) == 0:
    pred_df_tb = pd.DataFrame(columns=["datetime", "frequency_band", "actual", "predicted"])
else:
    pred_df_tb["datetime"] = pd.to_datetime(pred_df_tb["datetime"])
print("Sample:")
print(pred_df_tb.head(10).to_string(index=False))
print("...")
print(pred_df_tb.tail(5).to_string(index=False))

if len(pred_df_tb) > 0:
    y_true_tb = pred_df_tb["actual"].values
    y_pred_tb = pred_df_tb["predicted"].values
    mae_tb = float(np.mean(np.abs(y_true_tb - y_pred_tb)))
    rmse_tb = float(np.sqrt(np.mean((y_true_tb - y_pred_tb) ** 2)))
    _mase_diffs = pred_df_tb.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
    mase_tb = float(mae_tb / mase_denom) if mase_denom and mase_denom > 0 else np.nan
else:
    mae_tb = rmse_tb = mase_tb = None

print(f"\nMAE (tuned blend):  {mae_tb:.4f}" if mae_tb is not None else "\nMAE (tuned blend): —")
print(f"RMSE (tuned blend): {rmse_tb:.4f}" if rmse_tb is not None else "RMSE (tuned blend): —")
print(f"MASE (tuned blend): {mase_tb:.4f}" if mase_tb is not None else "MASE (tuned blend): —")
if mae_tb is not None and "mae_naive" in dir():
    diff = mae_naive - mae_tb
    print(f"  vs naive (MAE {mae_naive:.4f}): improvement {diff:.4f}" + (" — beats naive!" if diff > 0 else ""))

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "tuned_blend_metrics.json", "w") as f:
    json.dump({"mae": mae_tb, "rmse": rmse_tb, "mase": mase_tb, "n_predictions": len(pred_df_tb)}, f, indent=2)
pred_df_tb.to_parquet(out_dir / "tuned_blend_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'tuned_blend_metrics.json'}, {out_dir / 'tuned_blend_predictions.parquet'}")

Tuned blend (validation-tuned α): ŷ = α·y(t-1) + (1-α)·y(t-24), α in [0.97, 1.0] per band.

Sample:
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  30.409459
2025-01-19 01:00:00    3.100-3.105 12.972973  15.056757
2025-01-19 02:00:00    3.100-3.105 10.337838  12.893919
2025-01-19 03:00:00    3.100-3.105  9.932432  10.289189
2025-01-19 04:00:00    3.100-3.105  8.716216   9.932432
2025-01-19 05:00:00    3.100-3.105 13.581081   8.807432
2025-01-19 06:00:00    3.100-3.105 16.013514  13.538514
2025-01-19 07:00:00    3.100-3.105 27.770270  15.940541
2025-01-19 08:00:00    3.100-3.105 28.581081  27.612162
2025-01-19 09:00:00    3.100-3.105 42.972973  28.720946
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  77.981982
2026-01-18 20:00:00    3.445-3.450 57.117117  78.111712
2026-01-18 21:00:00    3.445-3.450 79.099099  57.127928
2026-01-18 22:00:00    3.445-3.450 79.099099  78.196396


In [116]:
# Seasonal weighted average: past N days at same time. ŷ(t) = weighted mean of y(t-24), y(t-48), ..., y(t-N*24)
print("Seasonal weighted average: prediction = weighted mean of same hour on past N days.")
print("Formula: ŷ(t) = sum_i w_i * y(t - 24*i) / sum(w_i), i = 1..N (more recent day = higher weight).\n")

TEST_STEPS_SW = 24 * 365
DAYS_SAME_TIME = 7  # past 7 days at same hour
# Weights: day 1 (yesterday) = 7, day 2 = 6, ..., day 7 = 1 (more recent = higher weight)
weights = np.arange(DAYS_SAME_TIME, 0, -1, dtype=float)
weights = weights / weights.sum()
rows_sw = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    need = TEST_STEPS_SW + DAYS_SAME_TIME * 24
    if len(sub) < need:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    for i in range(-TEST_STEPS_SW, 0):
        past = np.array([vals[i - 24 * (k + 1)] for k in range(DAYS_SAME_TIME)], dtype=float)
        mask = ~np.isnan(past)
        if np.any(mask):
            pred = float(np.sum(weights[mask] * past[mask]) / np.sum(weights[mask]))
        else:
            pred = np.nan
        rows_sw.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_sw = pd.DataFrame(rows_sw)
pred_df_sw["datetime"] = pd.to_datetime(pred_df_sw["datetime"])
pred_df_sw = pred_df_sw.dropna(subset=["predicted"])
print(f"Past {DAYS_SAME_TIME} days at same time (weights: recent=high). Sample:")
print(pred_df_sw.head(10).to_string(index=False))
print("...")
print(pred_df_sw.tail(5).to_string(index=False))

y_true_sw = pred_df_sw["actual"].values
y_pred_sw = pred_df_sw["predicted"].values
mae_sw = float(np.mean(np.abs(y_true_sw - y_pred_sw)))
rmse_sw = float(np.sqrt(np.mean((y_true_sw - y_pred_sw) ** 2)))
_mase_diffs = pred_df_sw.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_sw = float(mae_sw / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (seasonal wavg):  {mae_sw:.4f}")
print(f"RMSE (seasonal wavg): {rmse_sw:.4f}")
print(f"MASE (seasonal wavg): {mase_sw:.4f}" if not np.isnan(mase_sw) else "MASE (seasonal wavg): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_sw = {"mae": mae_sw, "rmse": rmse_sw, "mase": mase_sw, "n_predictions": len(pred_df_sw), "test_steps_hours": TEST_STEPS_SW, "days_same_time": DAYS_SAME_TIME}
with open(out_dir / "seasonal_wavg_metrics.json", "w") as f:
    json.dump(metrics_sw, f, indent=2)
pred_df_sw.to_parquet(out_dir / "seasonal_wavg_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'seasonal_wavg_metrics.json'}, {out_dir / 'seasonal_wavg_predictions.parquet'}")

sw_freqs = sorted(pred_df_sw["frequency_band"].unique())
fig_metrics_sw = go.Figure()
fig_metrics_sw.add_trace(go.Bar(x=["MAE", "RMSE", "MASE"], y=[mae_sw, rmse_sw, mase_sw if not np.isnan(mase_sw) else 0], text=[f"{mae_sw:.4f}", f"{rmse_sw:.4f}", f"{mase_sw:.4f}" if not np.isnan(mase_sw) else "—"], textposition="outside", marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"]))
fig_metrics_sw.update_layout(title="Seasonal weighted avg: MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_sw = dash.Dash(__name__, title="Seasonal weighted avg: actual vs predicted")
app_sw.layout = html.Div([
    html.Div([html.Label("Frequency band:"), dcc.Dropdown(id="sw-freq", options=[{"label": f, "value": f} for f in sw_freqs], value=sw_freqs[0], clearable=False, style={"width": "300px"})], style={"margin": "10px"}),
    dcc.Graph(id="sw-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}), dcc.Graph(id="sw-metrics-bar", figure=fig_metrics_sw),
])
@app_sw.callback(Output("sw-plot", "figure"), Input("sw-freq", "value"))
def update_sw_plot(freq):
    sub = pred_df_sw[pred_df_sw["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0: return go.Figure()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(title=f"Seasonal weighted avg ({DAYS_SAME_TIME}d): actual vs predicted — {freq}", xaxis_title="Date & time", yaxis_title="Value", xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)), yaxis=dict(autorange=True, rangemode="normal"), margin=dict(b=80, t=60), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), hovermode="x unified")
    return fig
app_sw.run(port=8059, jupyter_mode="inline", jupyter_height=900)

Seasonal weighted average: prediction = weighted mean of same hour on past N days.
Formula: ŷ(t) = sum_i w_i * y(t - 24*i) / sum(w_i), i = 1..N (more recent day = higher weight).

Past 7 days at same time (weights: recent=high). Sample:
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  12.299710
2025-01-19 01:00:00    3.100-3.105 12.972973  10.062741
2025-01-19 02:00:00    3.100-3.105 10.337838   9.729730
2025-01-19 03:00:00    3.100-3.105  9.932432  13.204633
2025-01-19 04:00:00    3.100-3.105  8.716216  13.269788
2025-01-19 05:00:00    3.100-3.105 13.581081  16.462355
2025-01-19 06:00:00    3.100-3.105 16.013514  18.692085
2025-01-19 07:00:00    3.100-3.105 27.770270  23.079151
2025-01-19 08:00:00    3.100-3.105 28.581081  28.472490
2025-01-19 09:00:00    3.100-3.105 42.972973  31.274131
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  50.805153
2026-01-18 20:00:00    3.445-3.

## Ideas to beat naive (experiments)

**Tweaks to current models**
- **Tuned blend with more historical data:** Use a longer validation window (e.g. 28 days instead of 14) so α is estimated with more data; or use the full train set to pick α (risk of overfitting).
- **Weekly seasonality:** There may be a day-of-week effect (e.g. weekends vs weekdays). Add **y(t−168)** (same hour, same weekday last week) and try a **three-way blend:** ŷ = α·y(t−1) + β·y(t−24) + γ·y(t−168) with α+β+γ=1, tuned on validation.
- **Per-hour α:** Use a different α for each hour-of-day (0..23) so the mix of naive vs seasonal adapts by time of day (already tried in tmp; can add to notebook).

**Other professional / ML models**
- **Chronos 1-step:** The notebook uses Chronos for 72h-ahead windows; we could run Chronos with `prediction_length=1` for true 1-step-ahead and compare MAE (same evaluation as naive).
- **Prophet:** Fit Prophet per band on train, then 1-step-ahead predict on test (Prophet is good at trend + daily/weekly seasonality).
- **Ridge / linear model with lags:** Features: y(t−1), y(t−24), y(t−168), plus optional hour_of_day, day_of_week. Fit Ridge on train (or validation), predict test. Simple ML that can capture weekly + daily effects.
- **LightGBM / XGBoost:** Same lag + calendar features; tree models can capture non-linear interactions (e.g. “weekend at 3am”).
- **SARIMA:** Seasonal ARIMA with period 24 (and optionally 168) — more parameters to tune but designed for seasonal series.

The cells below implement: **(1) Three-way blend** (daily + weekly seasonality), **(2) Tuned blend with longer validation**, **(3) Ridge with lags** (y(t−1), y(t−24), y(t−168)).

In [ ]:
# Three-way blend: ŷ = α·y(t-1) + β·y(t-24) + γ·y(t-168), α+β+γ=1. Tune (α,β,γ) per band on validation.
# Weekly seasonality: y(t-168) = same hour, same weekday last week.
print("Three-way blend (daily + weekly seasonality): ŷ = α·y(t-1) + β·y(t-24) + γ·y(t-168), α+β+γ=1.\n")

TEST_STEPS_3W = 24 * 365
VAL_STEPS_3W = 24 * 14
# Need at least 168 hours before validation end for y(t-168)
rows_3w = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    n = len(sub)
    if n < TEST_STEPS_3W + VAL_STEPS_3W + 168:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    val_start = n - TEST_STEPS_3W - VAL_STEPS_3W
    val_end = n - TEST_STEPS_3W
    if val_end - 168 <= val_start:
        continue
    # Grid search: α in [0.85, 1.0], then β, γ = (1-α)*[0,1] split (so γ = (1-α)*g, β = (1-α)*(1-g))
    best_mae = np.inf
    best_alpha, best_beta, best_gamma = 1.0, 0.0, 0.0
    for alpha in np.linspace(0.88, 1.0, 13):
        rest = 1.0 - alpha
        for g in np.linspace(0, 1, 9):  # gamma = rest*g, beta = rest*(1-g)
            gamma = rest * g
            beta = rest * (1 - g)
            preds = alpha * vals[val_start + 167 : val_end - 1] + beta * vals[val_start + 144 : val_end - 24] + gamma * vals[val_start : val_end - 168]
            actuals = vals[val_start + 168 : val_end]
            mae = np.mean(np.abs(actuals - preds))
            if mae < best_mae:
                best_mae = mae
                best_alpha, best_beta, best_gamma = alpha, beta, gamma
    for i in range(-TEST_STEPS_3W, 0):
        pred = best_alpha * vals[i - 1] + best_beta * vals[i - 24] + best_gamma * vals[i - 168]
        rows_3w.append({"datetime": dts[i], "frequency_band": freq, "actual": float(vals[i]), "predicted": float(pred)})
pred_df_3w = pd.DataFrame(rows_3w)
if len(pred_df_3w) > 0:
    pred_df_3w["datetime"] = pd.to_datetime(pred_df_3w["datetime"])
    mae_3w = float((pred_df_3w["actual"] - pred_df_3w["predicted"]).abs().mean())
    rmse_3w = float(np.sqrt(((pred_df_3w["actual"] - pred_df_3w["predicted"]) ** 2).mean()))
    _m = pred_df_3w.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    mase_3w = mae_3w / float(_m.mean()) if len(_m) > 0 else np.nan
else:
    mae_3w = rmse_3w = mase_3w = None
print(f"MAE (three-way blend):  {mae_3w:.4f}" if mae_3w is not None else "MAE (three-way blend): —")
print(f"RMSE (three-way blend): {rmse_3w:.4f}" if rmse_3w is not None else "RMSE: —")
print(f"MASE (three-way blend): {mase_3w:.4f}" if mase_3w is not None else "MASE: —")
if mae_3w is not None and "mae_naive" in dir():
    print(f"  vs naive (MAE {mae_naive:.4f}): improvement {mae_naive - mae_3w:.4f}" + (" — beats naive!" if mae_naive - mae_3w > 0 else ""))
out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
if pred_df_3w is not None and len(pred_df_3w) > 0:
    with open(out_dir / "three_way_blend_metrics.json", "w") as f:
        json.dump({"mae": mae_3w, "rmse": rmse_3w, "mase": mase_3w, "n_predictions": len(pred_df_3w)}, f, indent=2)
    pred_df_3w.to_parquet(out_dir / "three_way_blend_predictions.parquet", index=False)
    print(f"\nStored: {out_dir / 'three_way_blend_metrics.json'}, three_way_blend_predictions.parquet")

In [156]:
# Tuned blend with LONGER validation (28 days instead of 14). Same formula: ŷ = α·y(t-1) + (1-α)·y(t-24).
print("Tuned blend (longer validation, 28 days): α in [0.97, 1.0] per band.\n")

TEST_STEPS_TBL = 24 * 365
VAL_STEPS_TBL = 24 * 28   # 28 days validation
TRAIN_MIN_TBL = 24 * 28
rows_tbl = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    n = len(sub)
    if n < TEST_STEPS_TBL + TRAIN_MIN_TBL + 24:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    val_start = n - TEST_STEPS_TBL - VAL_STEPS_TBL
    val_end = n - TEST_STEPS_TBL
    if val_end - 24 <= val_start:
        continue
    best_mae = np.inf
    best_alpha = 1.0
    for alpha in np.linspace(0.97, 1.0, 31):
        preds = alpha * vals[val_start + 23 : val_end - 1] + (1 - alpha) * vals[val_start : val_end - 24]
        actuals = vals[val_start + 24 : val_end]
        mae = np.mean(np.abs(actuals - preds))
        if mae < best_mae:
            best_mae = mae
            best_alpha = alpha
    for i in range(-TEST_STEPS_TBL, 0):
        pred = best_alpha * vals[i - 1] + (1 - best_alpha) * vals[i - 24]
        rows_tbl.append({"datetime": dts[i], "frequency_band": freq, "actual": float(vals[i]), "predicted": float(pred)})
pred_df_tbl = pd.DataFrame(rows_tbl)
if len(pred_df_tbl) > 0:
    pred_df_tbl["datetime"] = pd.to_datetime(pred_df_tbl["datetime"])
    mae_tbl = float((pred_df_tbl["actual"] - pred_df_tbl["predicted"]).abs().mean())
    rmse_tbl = float(np.sqrt(((pred_df_tbl["actual"] - pred_df_tbl["predicted"]) ** 2).mean()))
    _m = pred_df_tbl.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    mase_tbl = mae_tbl / float(_m.mean()) if len(_m) > 0 else np.nan
else:
    mae_tbl = rmse_tbl = mase_tbl = None
print(f"MAE (tuned blend, 28d val):  {mae_tbl:.4f}" if mae_tbl is not None else "MAE: —")
print(f"RMSE: {rmse_tbl:.4f}" if rmse_tbl is not None else "RMSE: —")
print(f"MASE: {mase_tbl:.4f}" if mase_tbl is not None else "MASE: —")
if mae_tbl is not None and "mae_naive" in dir():
    print(f"  vs naive: improvement {mae_naive - mae_tbl:.4f}" + (" — beats naive!" if mae_naive - mae_tbl > 0 else ""))
if pred_df_tbl is not None and len(pred_df_tbl) > 0:
    out_dir = _root / "data" / "time_series"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "tuned_blend_long_val_metrics.json", "w") as f:
        json.dump({"mae": mae_tbl, "rmse": rmse_tbl, "mase": mase_tbl, "n_predictions": len(pred_df_tbl), "val_days": 28}, f, indent=2)
    pred_df_tbl.to_parquet(out_dir / "tuned_blend_long_val_predictions.parquet", index=False)
    print(f"\nStored: tuned_blend_long_val_metrics.json, tuned_blend_long_val_predictions.parquet")

Tuned blend (longer validation, 28 days): α in [0.97, 1.0] per band.

MAE: —
RMSE: —
MASE: —


In [155]:
# Ridge regression with lags: features y(t-1), y(t-24), y(t-168). Fit on train, predict test. Optional: hour_of_day, day_of_week.
print("Ridge with lags: ŷ = b0 + b1·y(t-1) + b2·y(t-24) + b3·y(t-168). Fit on train (last year excluded), predict test.\n")

from numpy.linalg import LinAlgError
TEST_STEPS_R = 24 * 365
TRAIN_MIN_R = 24 * 30 + 168  # need at least 30 days + 1 week for lags
rows_r = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    n = len(sub)
    if n < TEST_STEPS_R + TRAIN_MIN_R:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    train_end = n - TEST_STEPS_R
    # Train: y = b0 + b1*x1 + b2*x24 + b3*x168
    y_tr = vals[168:train_end]
    x1 = vals[167:train_end - 1]
    x24 = vals[144:train_end - 24]
    x168 = vals[0:train_end - 168]
    X_tr = np.column_stack([np.ones_like(y_tr), x1, x24, x168])
    lam = 1e-4
    try:
        beta = np.linalg.solve(X_tr.T @ X_tr + lam * np.eye(4), X_tr.T @ y_tr)
    except LinAlgError:
        beta = np.array([0.0, 1.0, 0.0, 0.0])
    for i in range(-TEST_STEPS_R, 0):
        pred = beta[0] + beta[1] * vals[i - 1] + beta[2] * vals[i - 24] + beta[3] * vals[i - 168]
        rows_r.append({"datetime": dts[i], "frequency_band": freq, "actual": float(vals[i]), "predicted": float(pred)})
pred_df_r = pd.DataFrame(rows_r)
if len(pred_df_r) > 0:
    pred_df_r["datetime"] = pd.to_datetime(pred_df_r["datetime"])
    mae_r = float((pred_df_r["actual"] - pred_df_r["predicted"]).abs().mean())
    rmse_r = float(np.sqrt(((pred_df_r["actual"] - pred_df_r["predicted"]) ** 2).mean()))
    _m = pred_df_r.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    mase_r = mae_r / float(_m.mean()) if len(_m) > 0 else np.nan
else:
    mae_r = rmse_r = mase_r = None
print(f"MAE (Ridge lags):  {mae_r:.4f}" if mae_r is not None else "MAE: —")
print(f"RMSE: {rmse_r:.4f}" if rmse_r is not None else "RMSE: —")
print(f"MASE: {mase_r:.4f}" if mase_r is not None else "MASE: —")
if mae_r is not None and "mae_naive" in dir():
    print(f"  vs naive: improvement {mae_naive - mae_r:.4f}" + (" — beats naive!" if mae_naive - mae_r > 0 else ""))
if pred_df_r is not None and len(pred_df_r) > 0:
    out_dir = _root / "data" / "time_series"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "ridge_lags_metrics.json", "w") as f:
        json.dump({"mae": mae_r, "rmse": rmse_r, "mase": mase_r, "n_predictions": len(pred_df_r)}, f, indent=2)
    pred_df_r.to_parquet(out_dir / "ridge_lags_predictions.parquet", index=False)
    print(f"\nStored: ridge_lags_metrics.json, ridge_lags_predictions.parquet")

Ridge with lags: ŷ = b0 + b1·y(t-1) + b2·y(t-24) + b3·y(t-168). Fit on train (last year excluded), predict test.

MAE: —
RMSE: —
MASE: —


In [117]:
# Seasonal exponential smoothing: past 6 days at same time. ES over (y(t-144), y(t-120), ..., y(t-24)).
print("Seasonal exponential smoothing: past 6 days at same hour, smoothed (recent days weighted more).")
print("Formula: level_1 = y(t-6d), level_k = α * y(t-(7-k)d) + (1-α)*level_{k-1}, ŷ = level_6.\n")

TEST_STEPS_SES = 24 * 365
DAYS_PAST_SES = 6  # past 6 days at same time
ALPHA_SES = 0.5  # smoothing over the 6 daily values (tunable later)
rows_ses = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    need = TEST_STEPS_SES + DAYS_PAST_SES * 24
    if len(sub) < need:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    for i in range(-TEST_STEPS_SES, 0):
        # Values from oldest to most recent: y(t-6*24), y(t-5*24), ..., y(t-24)
        past_vals = np.array([vals[i - 24 * (DAYS_PAST_SES - k)] for k in range(DAYS_PAST_SES)], dtype=float)
        if np.any(~np.isnan(past_vals)):
            level = np.nan
            for j, v in enumerate(past_vals):
                if np.isnan(v):
                    continue
                if np.isnan(level):
                    level = v
                else:
                    level = ALPHA_SES * v + (1 - ALPHA_SES) * level
            pred = float(level) if not np.isnan(level) else np.nan
        else:
            pred = np.nan
        rows_ses.append({"datetime": dts[i], "frequency_band": freq, "actual": vals[i], "predicted": pred})

pred_df_ses = pd.DataFrame(rows_ses)
pred_df_ses["datetime"] = pd.to_datetime(pred_df_ses["datetime"])
pred_df_ses = pred_df_ses.dropna(subset=["predicted"])
print(f"Past {DAYS_PAST_SES} days at same time, α={ALPHA_SES}. Sample:")
print(pred_df_ses.head(10).to_string(index=False))
print("...")
print(pred_df_ses.tail(5).to_string(index=False))

y_true_ses2 = pred_df_ses["actual"].values
y_pred_ses2 = pred_df_ses["predicted"].values
mae_ses2 = float(np.mean(np.abs(y_true_ses2 - y_pred_ses2)))
rmse_ses2 = float(np.sqrt(np.mean((y_true_ses2 - y_pred_ses2) ** 2)))
_mase_diffs = pred_df_ses.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_ses2 = float(mae_ses2 / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (seasonal ES):  {mae_ses2:.4f}")
print(f"RMSE (seasonal ES): {rmse_ses2:.4f}")
print(f"MASE (seasonal ES): {mase_ses2:.4f}" if not np.isnan(mase_ses2) else "MASE (seasonal ES): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_ses2 = {"mae": mae_ses2, "rmse": rmse_ses2, "mase": mase_ses2, "n_predictions": len(pred_df_ses), "test_steps_hours": TEST_STEPS_SES, "days_past": DAYS_PAST_SES, "alpha": ALPHA_SES}
with open(out_dir / "seasonal_es_metrics.json", "w") as f:
    json.dump(metrics_ses2, f, indent=2)
pred_df_ses.to_parquet(out_dir / "seasonal_es_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'seasonal_es_metrics.json'}, {out_dir / 'seasonal_es_predictions.parquet'}")

ses_freqs = sorted(pred_df_ses["frequency_band"].unique())
fig_metrics_ses2 = go.Figure()
fig_metrics_ses2.add_trace(go.Bar(x=["MAE", "RMSE", "MASE"], y=[mae_ses2, rmse_ses2, mase_ses2 if not np.isnan(mase_ses2) else 0], text=[f"{mae_ses2:.4f}", f"{rmse_ses2:.4f}", f"{mase_ses2:.4f}" if not np.isnan(mase_ses2) else "—"], textposition="outside", marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"]))
fig_metrics_ses2.update_layout(title="Seasonal ES (6d): MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_ses2 = dash.Dash(__name__, title="Seasonal ES: actual vs predicted")
app_ses2.layout = html.Div([
    html.Div([html.Label("Frequency band:"), dcc.Dropdown(id="ses2-freq", options=[{"label": f, "value": f} for f in ses_freqs], value=ses_freqs[0], clearable=False, style={"width": "300px"})], style={"margin": "10px"}),
    dcc.Graph(id="ses2-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}), dcc.Graph(id="ses2-metrics-bar", figure=fig_metrics_ses2),
])
@app_ses2.callback(Output("ses2-plot", "figure"), Input("ses2-freq", "value"))
def update_ses2_plot(freq):
    sub = pred_df_ses[pred_df_ses["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0: return go.Figure()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(title=f"Seasonal ES ({DAYS_PAST_SES}d, α={ALPHA_SES}): actual vs predicted — {freq}", xaxis_title="Date & time", yaxis_title="Value", xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)), yaxis=dict(autorange=True, rangemode="normal"), margin=dict(b=80, t=60), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), hovermode="x unified")
    return fig
app_ses2.run(port=8060, jupyter_mode="inline", jupyter_height=900)

Seasonal exponential smoothing: past 6 days at same hour, smoothed (recent days weighted more).
Formula: level_1 = y(t-6d), level_k = α * y(t-(7-k)d) + (1-α)*level_{k-1}, ŷ = level_6.

Past 6 days at same time, α=0.5. Sample:
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703  13.828125
2025-01-19 01:00:00    3.100-3.105 12.972973   9.932432
2025-01-19 02:00:00    3.100-3.105 10.337838   9.742399
2025-01-19 03:00:00    3.100-3.105  9.932432  10.831926
2025-01-19 04:00:00    3.100-3.105  8.716216  11.142314
2025-01-19 05:00:00    3.100-3.105 13.581081  13.631757
2025-01-19 06:00:00    3.100-3.105 16.013514  14.879645
2025-01-19 07:00:00    3.100-3.105 27.770270  19.092061
2025-01-19 08:00:00    3.100-3.105 28.581081  26.161318
2025-01-19 09:00:00    3.100-3.105 42.972973  31.621622
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739  54.206308
2026-01-18 20:00:00    3.445-3.450 57.1171

### Why naive keeps winning (and what it means)

The data is **not random** — it's highly **autocorrelated at lag 1**: the value one hour ago is an excellent predictor of the next hour. So the best simple rule for this series is “repeat the last value.”

- **Seasonal models** (yesterday same hour) lose because **hour-to-hour persistence is stronger than day-to-day similarity** at the same hour: y(t−1) carries more information than y(t−24) for your series.
- **Moving average** and **exponential smoothing** dilute the last observation slightly, so they lose to pure naive.
- **Hybrid** (blend of naive + seasonal) adds some y(t−24), which on average adds more noise than signal, so naive still wins.

**Interpretation:** Failing to beat naive does **not** mean the series is random. It means the **optimal simple predictor for 1-step-ahead is “last value.”** To do better you typically need (a) external features, (b) a model that learns when to trust last value vs seasonality, or (c) a longer horizon where seasonality might dominate. Below we try a **data-driven OLS blend** and a **median ensemble** to see if they can edge past naive.

In [138]:
# Seasonal ARIMA (SARIMA): season = 1 day (24 hours). Rolling 1-step-ahead.
print("Seasonal ARIMA: SARIMA with daily season (s=24). Rolling 1-step-ahead.\n")

from statsmodels.tsa.statespace.sarimax import SARIMAX

if "datetime" not in df.columns:
    df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")
TEST_STEPS_SARIMA = 24 * 1   # last 1 day (tuned for speed; increase to 24*3 for more points)
SARIMA_ORDER = (1, 0, 1)
SARIMA_SEASONAL = (1, 0, 1, 24)  # daily season
SARIMA_MAX_BANDS = 5   # tuned for speed (was 15)
SARIMA_TRAIN_WINDOW = 336  # fixed window: last 2 weeks — much faster than expanding
bands_sarima = (freq_bands[:SARIMA_MAX_BANDS] if SARIMA_MAX_BANDS else freq_bands)
rows_sarima = []
_first_err = None
for freq in bands_sarima:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS_SARIMA + SARIMA_TRAIN_WINDOW + 24:
        continue
    dts = sub["datetime"].values
    vals = sub[freq].values.astype(float)
    n = len(vals)
    for i in range(n - TEST_STEPS_SARIMA, n - 1):
        start = max(0, i + 1 - SARIMA_TRAIN_WINDOW)
        train_series = pd.Series(vals[start : i + 1]).dropna()
        if len(train_series) < 24 * 2 + 50:
            continue
        try:
            model = SARIMAX(train_series, order=SARIMA_ORDER, seasonal_order=SARIMA_SEASONAL)
            fitted = model.fit(disp=False)
            f = fitted.get_forecast(steps=1)
            pred = float(f.predicted_mean.iloc[0])
        except Exception as e:
            if _first_err is None:
                _first_err = e
            pred = np.nan
        rows_sarima.append({"datetime": dts[i + 1], "frequency_band": freq, "actual": vals[i + 1], "predicted": pred})

if _first_err is not None and len(rows_sarima) == 0:
    print(f"SARIMA failed for all bands. First error: {_first_err}")
pred_df_sarima = pd.DataFrame(rows_sarima)
if len(pred_df_sarima) > 0:
    pred_df_sarima["datetime"] = pd.to_datetime(pred_df_sarima["datetime"])
else:
    pred_df_sarima = pd.DataFrame(columns=["datetime", "frequency_band", "actual", "predicted"])

print(f"Order {SARIMA_ORDER}, seasonal {SARIMA_SEASONAL}. Sample:")
print(pred_df_sarima.head(10).to_string(index=False))
print("...")
print(pred_df_sarima.tail(5).to_string(index=False))

valid_sarima = pred_df_sarima.dropna(subset=["predicted"])
y_true_sarima = valid_sarima["actual"].values
y_pred_sarima = valid_sarima["predicted"].values
mae_sarima = float(np.mean(np.abs(y_true_sarima - y_pred_sarima))) if len(valid_sarima) > 0 else np.nan
rmse_sarima = float(np.sqrt(np.mean((y_true_sarima - y_pred_sarima) ** 2))) if len(valid_sarima) > 0 else np.nan
_mase_diffs = valid_sarima.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_sarima = float(mae_sarima / mase_denom) if (mase_denom and mase_denom > 0 and not np.isnan(mae_sarima)) else np.nan

print(f"\nMAE (SARIMA):  {mae_sarima:.4f}" if not np.isnan(mae_sarima) else "\nMAE (SARIMA):  (no valid predictions)")
print(f"RMSE (SARIMA): {rmse_sarima:.4f}" if not np.isnan(rmse_sarima) else "RMSE (SARIMA): (no valid predictions)")
print(f"MASE (SARIMA): {mase_sarima:.4f}" if not np.isnan(mase_sarima) else "MASE (SARIMA): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_sarima = {"mae": mae_sarima, "rmse": rmse_sarima, "mase": mase_sarima, "n_predictions": len(valid_sarima), "test_steps_hours": TEST_STEPS_SARIMA, "order": list(SARIMA_ORDER), "seasonal_order": list(SARIMA_SEASONAL)}
with open(out_dir / "sarima_metrics.json", "w") as f:
    json.dump(metrics_sarima, f, indent=2)
pred_df_sarima.to_parquet(out_dir / "sarima_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'sarima_metrics.json'}, {out_dir / 'sarima_predictions.parquet'}")

sarima_freqs = sorted(pred_df_sarima["frequency_band"].unique()) if len(pred_df_sarima) > 0 else ["(none)"]
fig_metrics_sarima = go.Figure()
fig_metrics_sarima.add_trace(go.Bar(x=["MAE", "RMSE", "MASE"], y=[mae_sarima if not np.isnan(mae_sarima) else 0, rmse_sarima if not np.isnan(rmse_sarima) else 0, mase_sarima if not np.isnan(mase_sarima) else 0], text=[f"{mae_sarima:.4f}" if not np.isnan(mae_sarima) else "—", f"{rmse_sarima:.4f}" if not np.isnan(rmse_sarima) else "—", f"{mase_sarima:.4f}" if not np.isnan(mase_sarima) else "—"], textposition="outside", marker_color=["#1f77b4", "#ff7f0e", "#2ca02c"]))
fig_metrics_sarima.update_layout(title="SARIMA (s=24): MAE, RMSE, MASE", xaxis_title="Metric", yaxis_title="Value", margin=dict(b=60, t=50), height=350)

app_sarima = dash.Dash(__name__, title="SARIMA: actual vs predicted")
app_sarima.layout = html.Div([
    html.Div([html.Label("Frequency band:"), dcc.Dropdown(id="sarima-freq", options=[{"label": f, "value": f} for f in sarima_freqs], value=sarima_freqs[0], clearable=False, style={"width": "300px"})], style={"margin": "10px"}),
    dcc.Graph(id="sarima-plot", style={"height": "450px"}),
    html.H3("MAE, RMSE, MASE", style={"marginTop": "20px"}), dcc.Graph(id="sarima-metrics-bar", figure=fig_metrics_sarima),
])
@app_sarima.callback(Output("sarima-plot", "figure"), Input("sarima-freq", "value"))
def update_sarima_plot(freq):
    if freq is None or (len(pred_df_sarima) == 0): return go.Figure()
    sub = pred_df_sarima[pred_df_sarima["frequency_band"] == freq].sort_values("datetime")
    if len(sub) == 0: return go.Figure()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["actual"], mode="lines+markers", name="Actual", line=dict(width=2, color="black")))
    fig.add_trace(go.Scatter(x=sub["datetime"], y=sub["predicted"], mode="lines+markers", name="Predicted", line=dict(width=1.5, color="blue", dash="dash")))
    fig.update_layout(title=f"SARIMA {SARIMA_ORDER} {SARIMA_SEASONAL}: actual vs predicted — {freq}", xaxis_title="Date & time", yaxis_title="Value", xaxis=dict(type="date", rangeslider=dict(visible=True, thickness=0.05)), yaxis=dict(autorange=True, rangemode="normal"), margin=dict(b=80, t=60), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1), hovermode="x unified")
    return fig
app_sarima.run(port=8061, jupyter_mode="inline", jupyter_height=900)

Seasonal ARIMA: SARIMA with daily season (s=24). Rolling 1-step-ahead.



/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals

/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals

/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals

/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals

/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retv

Order (1, 0, 1), seasonal (1, 0, 1, 24). Sample:
           datetime frequency_band    actual  predicted
2026-01-18 01:00:00    3.100-3.105 17.477477  20.829482
2026-01-18 02:00:00    3.100-3.105 17.477477  16.737003
2026-01-18 03:00:00    3.100-3.105 15.495495  17.122895
2026-01-18 04:00:00    3.100-3.105 16.036036  15.577820
2026-01-18 05:00:00    3.100-3.105 16.036036  16.061018
2026-01-18 06:00:00    3.100-3.105 16.036036  17.838764
2026-01-18 07:00:00    3.100-3.105 16.756757  18.164777
2026-01-18 08:00:00    3.100-3.105 21.981982  19.542790
2026-01-18 09:00:00    3.100-3.105 21.981982  21.650812
2026-01-18 10:00:00    3.100-3.105 26.486486  22.763272
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.120-3.125 74.594595  70.242088
2026-01-18 20:00:00    3.120-3.125 57.117117  73.085068
2026-01-18 21:00:00    3.120-3.125 57.117117  59.076646
2026-01-18 22:00:00    3.120-3.125 57.117117  53.351423
2026-01-18 23:00:00    3.120-3.125 75.135135  55.10

## Model comparison: Diebold-Mariano test

Compares forecast accuracy of two models on **aligned** prediction points.  
Null: equal predictive accuracy (MAE). Rejecting H₀ indicates one method is significantly more accurate.

### How to read each line of output

Each line is: **`  A vs B: n=N, DM=X.XXXX, p=0.XXXX *  (lower MAE: Z)`**

1. **`A vs B`** — We compare model **A** (first) and model **B** (second). Errors are **e1 = actual − predicted_A** and **e2 = actual − predicted_B**.

2. **`n=N`** — Number of (datetime, frequency_band) points where **both** models have a prediction. Only these aligned points are used. Larger n → more stable comparison.

3. **`DM=X.XXXX`** — Test statistic from the loss difference **d = |e1| − |e2|** (MAE loss).
   - **DM > 0** → On average the **first** model (A) has **larger** absolute errors → A is **worse** → **B** has lower MAE.
   - **DM < 0** → On average the **first** model (A) has **smaller** absolute errors → A is **better** → **A** has lower MAE.
   - **DM ≈ 0** → No clear difference.

4. **`p=0.XXXX`** — Two-sided p-value for H₀: “both models have equal accuracy.”
   - **p < 0.05** → We reject H₀ → the difference in MAE is **statistically significant** (not just noise).
   - **p ≥ 0.05** → We do not reject H₀ → we cannot say one is significantly better.

5. **`*`** — Shown only when **p < 0.05** (significant).

6. **`(lower MAE: Z)`** — **Z** is the model with the **smaller** MAE on this aligned set (the “winner”).
   - If **DM > 0** → first model worse → Z = **B**.
   - If **DM < 0** → first model better → Z = **A**.

**Example:** `naive vs exp_smooth: n=613200, DM=-82.83, p=0.0000 *  (lower MAE: naive)`  
→ We compare naive and exp_smooth on 613,200 aligned points. DM is **negative**, so the **first** model (naive) has smaller errors on average → **naive** has lower MAE. The difference is significant (p ≈ 0, *). The line correctly says “lower MAE: naive”.

In [165]:
# Diebold-Mariano test: compare two forecast error sequences (MAE loss)
# H0: equal predictive accuracy. d_t = |e1_t| - |e2_t|; DM = d_bar / (sigma_d / sqrt(n)); two-sided p-value.
from scipy import stats

def dm_test(e1, e2, loss="abs", h=1):
    """e1, e2: arrays of forecast errors (same length). loss='abs' => MAE. Returns (dm_stat, p_value_two_sided)."""
    e1, e2 = np.asarray(e1, dtype=float), np.asarray(e2, dtype=float)
    n = len(e1)
    if n != len(e2) or n < 2:
        return np.nan, np.nan
    if loss == "abs":
        d = np.abs(e1) - np.abs(e2)
    else:
        d = (e1 ** 2) - (e2 ** 2)  # squared error
    d_bar = np.mean(d)
    sigma_d = np.std(d, ddof=1)
    if sigma_d <= 0:
        return 0.0, 1.0
    dm_stat = d_bar / (sigma_d / np.sqrt(n))
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return float(dm_stat), float(p_value)

# Build aligned prediction data: merge on (datetime, frequency_band)
# Use in-memory pred_df, pred_df_ma, pred_df_es, pred_df_arima; pred_df_tf (Chronos) has different timestamps.
out_dir = _root / "data" / "time_series"
dfs = {}
for name, fname in [("naive", "naive_predictions.parquet"), ("moving_avg", "moving_avg_predictions.parquet"),
                    ("exp_smooth", "exp_smooth_predictions.parquet"), ("arima", "arima_predictions.parquet"),
                    ("chronos", "chronos_predictions.parquet"),
                    ("seasonal_naive", "seasonal_naive_predictions.parquet"), ("seasonal_wavg", "seasonal_wavg_predictions.parquet"),
                    ("seasonal_es", "seasonal_es_predictions.parquet"), ("sarima", "sarima_predictions.parquet"),
                    ("hybrid", "hybrid_predictions.parquet"), ("ols_blend", "ols_blend_predictions.parquet"), ("median_ensemble", "median_ensemble_predictions.parquet")]:
    p = out_dir / fname
    if p.exists():
        dfs[name] = pd.read_parquet(p)
        dfs[name]["datetime"] = pd.to_datetime(dfs[name]["datetime"])
    else:
        dfs[name] = None

# Pairwise DM tests on common (datetime, frequency_band) rows
model_pairs = [("naive", "moving_avg"), ("naive", "exp_smooth"), ("naive", "arima"), ("moving_avg", "exp_smooth"),
               ("moving_avg", "arima"), ("exp_smooth", "arima"),
               ("naive", "seasonal_naive"), ("naive", "seasonal_wavg"), ("naive", "seasonal_es"), ("naive", "sarima"),
               ("seasonal_naive", "seasonal_wavg"), ("seasonal_naive", "seasonal_es"), ("seasonal_naive", "sarima"),
               ("seasonal_wavg", "seasonal_es"), ("seasonal_wavg", "sarima"), ("seasonal_es", "sarima"),
               ("naive", "hybrid"), ("hybrid", "seasonal_naive"), ("hybrid", "exp_smooth")]
# Chronos vs others only where timestamps overlap
for m in ["naive", "moving_avg", "exp_smooth", "arima", "seasonal_naive", "seasonal_wavg", "seasonal_es", "sarima", "hybrid", "ols_blend", "median_ensemble"]:
    if dfs.get("chronos") is not None and dfs.get(m) is not None:
        model_pairs.append((m, "chronos"))

print("Diebold-Mariano test (MAE loss, two-sided). H0: equal accuracy.")
print("DM > 0 => first model worse; DM < 0 => first model better. p < 0.05 => significant.\n")

for a, b in model_pairs:
    da, db = dfs.get(a), dfs.get(b)
    if da is None or db is None or len(da) == 0 or len(db) == 0:
        continue
    m = da.merge(db, on=["datetime", "frequency_band"], suffixes=("_a", "_b"), how="inner")
    m = m.dropna(subset=["actual_a", "predicted_a", "actual_b", "predicted_b"])
    if len(m) < 10:
        print(f"  {a} vs {b}: too few aligned points ({len(m)}), skip")
        continue
    e1 = (m["actual_a"] - m["predicted_a"]).values
    e2 = (m["actual_b"] - m["predicted_b"]).values
    dm_stat, p_val = dm_test(e1, e2, loss="abs")
    # DM > 0 => first model (a) has larger |errors| => b has lower MAE; DM < 0 => a has lower MAE
    winner = b if dm_stat > 0 else a
    sig = " *" if p_val < 0.05 else ""
    print(f"  {a} vs {b}: n={len(m)}, DM={dm_stat:.4f}, p={p_val:.4f}{sig}  (lower MAE: {winner})")

Diebold-Mariano test (MAE loss, two-sided). H0: equal accuracy.
DM > 0 => first model worse; DM < 0 => first model better. p < 0.05 => significant.

  naive vs moving_avg: n=613200, DM=-124.9832, p=0.0000 *  (lower MAE: naive)
  naive vs exp_smooth: n=613200, DM=-65.2614, p=0.0000 *  (lower MAE: naive)
  naive vs arima: n=1065, DM=-11.8391, p=0.0000 *  (lower MAE: naive)
  moving_avg vs exp_smooth: n=613200, DM=126.5351, p=0.0000 *  (lower MAE: exp_smooth)
  moving_avg vs arima: n=1065, DM=4.0062, p=0.0001 *  (lower MAE: arima)
  exp_smooth vs arima: n=1065, DM=-11.5187, p=0.0000 *  (lower MAE: exp_smooth)
  naive vs seasonal_naive: n=613200, DM=-156.6232, p=0.0000 *  (lower MAE: naive)
  naive vs seasonal_wavg: n=613200, DM=-168.4609, p=0.0000 *  (lower MAE: naive)
  naive vs seasonal_es: n=613200, DM=-157.2944, p=0.0000 *  (lower MAE: naive)
  naive vs sarima: n=115, DM=-4.5299, p=0.0000 *  (lower MAE: naive)
  seasonal_naive vs seasonal_wavg: n=613200, DM=23.0343, p=0.0000 *  (lower

## Error calculation review (sanity check)

**Summary:** The notebook’s error logic is consistent; naive winning is plausible.

1. **Test period** — Naive, moving average, and exponential smoothing all use the same test window: last `TEST_STEPS = 24*365` hours. Same `(datetime, frequency_band)` set where all have predictions (same bands that pass the length check).

2. **Naive formula** — For each test index `i` we set `actual = vals[i]`, `predicted = vals[i-1]`, i.e. ŷ(t) = y(t-1). No look-ahead.

3. **MAE / RMSE** — `MAE = mean(|actual - predicted|)`, `RMSE = sqrt(mean((actual - predicted)^2))`. Standard definitions, applied over all rows in each model’s prediction table.

4. **MASE** — Denominator = mean of `|actual.diff()|` within each band (one-step changes in actuals on the **evaluation** set). So `MASE = MAE(model) / mean(|y_t - y_{t-1}|)`. For naive, `MAE_naive = mean(|y_t - y_{t-1}|)`, so **MASE_naive = 1** by construction. Other models use the same denominator definition on their own prediction tables; for models with the same eval set (naive, MA, ES, seasonal, hybrid, etc.) the denominator is the same, so MASE is comparable.

5. **Diebold–Mariano** — Merge on `(datetime, frequency_band)`; errors `e1 = actual - predicted_a`, `e2 = actual - predicted_b`; loss difference `d = |e1| - |e2|`; winner is the model with lower MAE on the aligned set. Logic is correct.

6. **Why naive can win** — For 1-step-ahead hourly data, the last observation y(t-1) is often very informative. Smoothing (MA, seasonal, etc.) averages over more history and can dilute that. So it’s reasonable that naive has the lowest MAE here.

In [140]:
# Sanity check: verify error calculations (run after naive + moving_avg cells have run and written parquet)
out_dir = _root / "data" / "time_series"
check = pd.read_parquet(out_dir / "naive_predictions.parquet")
check["datetime"] = pd.to_datetime(check["datetime"])
# MASE denominator = mean(|actual.diff()|) on eval set; for naive, MAE_naive = mean(|actual - predicted|) = same
mase_d = check.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs().mean()
mae_n = (check["actual"] - check["predicted"]).abs().mean()
print("Naive: MAE = mean(|actual - predicted|) =", round(mae_n, 6))
print("MASE denominator = mean(|actual.diff()|) =", round(mase_d, 6))
print("MASE_naive should be MAE/denom =", round(mae_n / mase_d, 6), "(expect 1.0)")
# Spot-check: for naive, predicted at t should equal actual at previous row (same band)
sorted_check = check.sort_values(["frequency_band", "datetime"])
prev_actual = sorted_check.groupby("frequency_band")["actual"].shift(1)
valid = prev_actual.notna()
ok = np.isclose(sorted_check.loc[valid, "predicted"], prev_actual.loc[valid])
print("Spot-check: predicted == previous actual (same band):", ok.sum(), "/", valid.sum(), "rows with prev (first row per band excluded)")
print("Done. If MASE_naive ≈ 1 and all spot-check rows ok, error logic is consistent.")

Naive: MAE = mean(|actual - predicted|) = 2.419918
MASE denominator = mean(|actual.diff()|) = 2.419819
MASE_naive should be MAE/denom = 1.000041 (expect 1.0)
Spot-check: predicted == previous actual (same band): 613130 / 613130 rows with prev (first row per band excluded)
Done. If MASE_naive ≈ 1 and all spot-check rows ok, error logic is consistent.
